# web item

In [ ]:
from __future__ import annotations
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *

import random
import math
import pandas as pd
import httpx
import re
import heapq
import json
import logging

import networkx as nx
from itertools import combinations

from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import string
from typing import NamedTuple


In [ ]:
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.overlay import OverlaySpec, FlowOverlay, CreamOverlay, RiverOverlay, OceanWaveOverlay
from HexMagic.overlay import ClimateOverlay, WatershedOverlay, TemperatureOverlay, PrecipitationOverlay,SoilOverlay
from HexMagic.climate import  TerraDemo, TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve:  dict[int, float] = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list[float] = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
class GameParts:

    def __init__(self, radius: int = 20, useKoreanMap: bool = False,size=300):
        if useKoreanMap:
            self.terr: TerraDemo = TerraDemo().japan_korea_map()
        else:
            world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=23,
                debug=True
            )

            self.terr = world.terrain
        self.grid: HexGrid = self.terr.hexGrid
        self.referenceIndex: int = self.grid.middle
        self.grid.adjustRadius(radius)
        rivers = self.terr.carve_to_ocean(num_lakes=0)
        self.builder: SVGBuilder = self.grid.builder
        self.basins: DrainageBasins = world.basins
        for i in range(len(self.grid.hexes)):
            self.grid.hexes[i].label = str(i)

        fy: FoodYield = FoodYield(self.terr, self.basins)
        
        self.yields: np.ndarray = fy.compute()
        self.fy: FoodYield = fy


In [ ]:

@patch
def hp2i(self: GameParts, hexpos: HexPosition) -> int:
    return self.grid.hexposition_to_index(hexpos=hexpos, origin_index=self.referenceIndex)

@patch
def i2hp(self: GameParts, index: int) -> HexPosition:
   return self.grid.index_to_hexposition(index=index, origin_index=self.referenceIndex)
        

In [ ]:
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    # Collect all squads and pieces, merging extras if provided
    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend(sq.alive if hasattr(sq, 'alive') else sq.pieces)
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
    terrain=terrain, grid=grid, builder=grid.builder,
    c2f=getattr(terrain, 'c2f', None),
    extras=dict(
        basins=parts.basins,
        coarse_basins=parts.basins,
        corridors=corridors or [],
        pieces=pieces,
        squads=squads,
    )
)

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts(radius=30)

In [ ]:
ctx = myStuff.overlayContext()
if showDemo:
    TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        FoodOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
ctx = myStuff.overlayContext()
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        

        ctx=ctx,
        debug = not showDemo
    )

## MapPlace

In [ ]:
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None
    tenure: int = 0  # turns spent at this location
    pantry: int = 0  # per-piece food storage


In [ ]:
@patch
def location(self:GameParts,location, facing = HexPosition.W):
    index = self.hp2i(location)
    retPlace=MapPlace(location,facing)
    retPlace.level = self.terr.elevationLevel(index)
    retPlace.harvest  = int(self.fy.tiers[index])
    retPlace.parent = self
    return retPlace

In [ ]:
@patch
def location_by_index(self:GameParts,index, facing = HexPosition.W):
    location = self.i2hp(index)
    return self.location(location,facing)

In [ ]:
myStuff.grid.middle, myStuff.grid.index_to_hexposition(myStuff.grid.middle), myStuff.i2hp(myStuff.grid.middle)

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.harvest

In [ ]:
@patch
def sight(self:MapPlace) -> HexRegion:
    """All hex indices visible from this piece's location."""
  
    effective_sight = max(2,self.level)
      
    retRegion = HexRegion(
        hexes=set([self.parent.hp2i(hp) for hp in self.location.field_of_view( self.facing, effective_sight)]),
        hexGrid = self.parent.grid)

    return retRegion


In [ ]:
place.sight()

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.sight()

In [ ]:
place = myStuff.location_by_index(273)
place.level, place.harvest

## Pieces

In [ ]:
LIFESPAN: dict[PieceType, int] = {
    PieceType.PAWN:   18,
    PieceType.BISHOP: 12,
    PieceType.KNIGHT: 3,
    PieceType.QUEEN:  24,
    PieceType.KING:   999,
}

# after the strategy
LIFESPAN[PieceType.PAWN]   = 6
LIFESPAN[PieceType.BISHOP] = 3
LIFESPAN[PieceType.KNIGHT] = 2
LIFESPAN[PieceType.QUEEN]  = 6
LIFESPAN[PieceType.KING]   = 999


In [ ]:
@dataclass
class Piece:
    id:str = ""
    flag:CountryFlag | None = None
    type:PieceType = PieceType.PAWN
    place:MapPlace | None = None
    timeout:int = 0
    name:str = ""
    year:int = 1900
    lantern = StyleCSS("blank",fill="#007",opacity=0.3)

In [ ]:
@patch
def __eq__(self: Piece, other) -> bool:
    if not isinstance(other, Piece): return False
    return self.id == other.id

@patch
def __hash__(self: Piece) -> int:
    return hash(self.id)


In [ ]:
@patch
def visible_allies(self: Piece, squad: 'Squad') -> list['Piece']:
    """Allies in this piece's sight cone, sorted by birth year then name."""
    if self.place is None:
        return []
    my_cone = self.sight().hexes
    parts = self.place.parent
    allies = []
    for p in squad.pieces:
        if p is self or p.place is None:
            continue
        idx = parts.hp2i(p.place.location)
        if idx in my_cone:
            allies.append(p)
    # Oldest first (lowest year), then alphabetically
    allies.sort(key=lambda p: (p.year, p.name))
    return allies


In [ ]:


@patch
def fetch_avatar(self: Piece, style: str = "avataaars") -> str:
    """Fetch and cache DiceBear SVG avatar using piece name as seed."""
    if not hasattr(self, '_avatar_svg') or self._avatar_svg is None:
        seed = self.name or f"piece_{id(self)}"
        resp = httpx.get(f"https://api.dicebear.com/9.x/{style}/svg?seed={seed}")
        resp.raise_for_status()
        self._avatar_svg = resp.text
    return self._avatar_svg

@patch
def avatar_id(self: Piece) -> str:
    return f"avatar_{self.name or id(self)}"

@patch
def register_avatar(self: Piece, builder: SVGBuilder, style: str = "avataaars"):
    """Register avatar as a <symbol> definition on the builder (idempotent)."""
    aid = self.avatar_id()
    # Skip if already registered
    if any(getattr(d, 'attributes', {}).get('id') == aid for d in builder.definitions):
        return
    svg_raw = self.fetch_avatar(style)
    # Extract viewBox from the SVG
    
    vb_match = re.search(r'viewBox="([^"]+)"', svg_raw)
    vb = vb_match.group(1) if vb_match else "0 0 280 280"
    # Strip outer <svg> tags to get inner content
    inner = re.sub(r'<svg[^>]*>', '', svg_raw)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    symbol = SVGDef("symbol", aid, inner, viewBox=vb)
    builder.add_definition(symbol)

@patch
def draw_avatar(self: Piece, center: MapCord, builder: SVGBuilder,
                size: float = 40, layer: str = None,
                opacity: float = 1.0, attrs: dict = None) -> str:
    """Render avatar at center with given size. Registers symbol if needed."""
    self.register_avatar(builder)
    aid = self.avatar_id()
    x, y = center.x - size / 2, center.y - size / 2
    extra = CountryFlag._render_attrs(attrs) if attrs else ""
    svg = (f'<use href="#{aid}" x="{x:.1f}" y="{y:.1f}" '
           f'width="{size:.1f}" height="{size:.1f}"{extra}/>')
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer:
        builder.adjust(layer, svg)
    return svg


In [ ]:
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int        # upkeep per turn while on board
    cost: int        # one-time cost to place
    harvest: float   # multiplier applied to tile's food yield tier
    drain: float     # cost imposed on EACH enemy piece in sight cone
    bonus_sight: int # extra sight range beyond elevation-based default

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=2, cost=7,  harvest=1.25, drain=0.25, bonus_sight=0),
    PieceType.BISHOP: FoodStats(diet=3, cost=14, harvest=0.75, drain=1.5,  bonus_sight=2),
    PieceType.KNIGHT: FoodStats(diet=2, cost=6,  harvest=0.0,  drain=3.0,  bonus_sight=0),
    PieceType.QUEEN:  FoodStats(diet=5, cost=27, harvest=0.0,  drain=4.25, bonus_sight=0),
    PieceType.KING:   FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1),
}

#This is after the new strategies to even them
FOOD_TABLE[PieceType.PAWN]   = FoodStats(diet=1, cost=3,  harvest=1.25, drain=1.0,  bonus_sight=0)
FOOD_TABLE[PieceType.BISHOP] = FoodStats(diet=3, cost=9,  harvest=1.0,  drain=0.75, bonus_sight=1)
FOOD_TABLE[PieceType.KNIGHT] = FoodStats(diet=2, cost=10, harvest=0.0,  drain=3.0,  bonus_sight=0)
FOOD_TABLE[PieceType.QUEEN]  = FoodStats(diet=4, cost=18, harvest=0.0,  drain=4.5,  bonus_sight=2)
FOOD_TABLE[PieceType.KING]   = FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1)


@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]


In [ ]:
@patch
def sight(self: Piece) -> HexRegion:
    """All hex indices visible from this piece's location."""
    bonus = self.food.bonus_sight
    effective_sight = max(2, self.place.level) + bonus
    
    return HexRegion(
        hexes=set([self.place.parent.hp2i(hp) 
                   for hp in self.place.location.field_of_view(self.place.facing, effective_sight)]),
        hexGrid=self.place.parent.grid)


## Squad

In [ ]:
@dataclass
class Squad:
    pieces: list[Piece] = field(default_factory=list)
    name: str = ""
    year: int = 1900
    animal: str = ""
    flag: CountryFlag | None = None
    countryName: str = ""
    storage:int = 100
    target:int = -1

    @classmethod
    def squads(cls, count: int = 4, flag: CountryFlag | None = None) -> list['Squad']:
        if flag is None:
            flag = CountryFlag.seaborn("husl", 1)[0]
        return [Squad(name=name, animal=animal, flag=flag) 
                for animal, name in flag.country_squads(count)]

In [ ]:
@patch
def make_squad(self: CountryFlag, piece_types: list[PieceType], year: int, seed: int = 0) -> Squad:
    """Create a Squad with animal name and age-appropriate named Pieces."""
    animal = self.animal_name()
    squad_nm = self.squad_name(animal, seed)
    
    rng = random.Random(hash((self.name, year, seed)))
    
    pieces = []
    for pt in piece_types:
        birth_year = year + rng.randint(20, 38)
        names = self.commonNames(year=birth_year)
        name = rng.choice(names)
        player = Piece(flag=self, type=pt, name=name, year=birth_year)
        player.id = ''.join(random.choices(string.ascii_letters, k=16))
        player.lantern = StyleCSS(f"{player.id}_lantern", fill=self.primary, opacity=0.3)
        pieces.append(player)

    
    return Squad(pieces=pieces, name=squad_nm, year=year, animal=animal, flag=self)


In [ ]:
@patch
def loadSquads(self:GameParts, year: int=1880, seed: int = 0,levels=3):
    self.flags = CountryFlag.seaborn("bright",levels=levels)
    self.squads = []
    for i,flag in enumerate(self.flags):
        roster = {
            PieceType.PAWN: 8, 
            PieceType.BISHOP: 4, 
            PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]


        squad = flag.make_squad(pieceKinds, year=year, seed =seed) 
        self.squads.append(squad)

In [ ]:
myStuff.loadSquads()

In [ ]:
myStuff.squads[0].pieces[0]

In [ ]:
from fasthtml.common import *
from fasthtml.components import Uk_input_tag
from fasthtml.svg import *
from monsterui.all import *

In [ ]:
read_url("https://monsterui.answer.ai/cards/md")

In [ ]:

@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    return Card(
        DivCentered(
            NotStr(b.xml()),
            P(self.name, cls=(TextT.sm, TextT.medium)),
            P(f"{self.type.icon} · {self.year}", cls=TextPresets.muted_sm)),
        cls=CardT.hover)


In [ ]:
myStuff.loadSquads()
show(myStuff.squads[0].pieces[0])

In [ ]:
#show(myStuff.squads[0].pieces[0])

In [ ]:
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    flag_builder = self.flag.flag_banner(width=210, height=120, wavy=True)

    # Animal icon
    ab = SVGBuilder()
    ab.width, ab.height = 250,250
    ab.adjust("animal", self.flag.animal_svg(
        size='large', animal=self.animal,
        center=MapCord(125, 125),
        piece_id=f"sq_{self.name}"))


    header = DivFullySpaced(
       # NotStr(flag_builder.xml()),
        H3(self.name),
        NotStr(ab.xml()),
        

        cls=TextPresets.muted_sm)

    # Group pieces by type
    from collections import defaultdict
    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        # Section header with piece icon
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm))
        # 4-wide grid of piece cards
        grid = Grid(*[p.__ft__() for p in pieces],
                    cols_sm=2, cols_md=4)
        sections.append(Div(sec_header, grid, cls='space-y-2'))

    return DivCentered(header,Card( *sections, cls='space-y-4'))


In [ ]:
@patch
def table(self: Squad):
    rows = []
    for p in self.pieces:
        loc = ""
        facing = ""
        if p.place and p.place.parent:
            loc = str(p.place.parent.hp2i(p.place.location))
            facing = str(p.place.facing.label)
        rows.append(Tr(
            Td(p.type.icon),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ))
    
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
            Tbody(*rows),
        ),
        header=(H4(self.name), P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)),
    )


In [ ]:
myStuff.loadSquads()
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
show(unit)

In [ ]:
unit.place.sight()

In [ ]:
unit.sight()

## Overlays

In [ ]:
def FogOverlay( fill="#ffffff", stroke="#cccccc", **kw) -> OverlaySpec:
    """White out hexes with no terrain data."""
    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        if not c2f: return ""
        mapped = {ni for indices in c2f.values() for ni in (indices if isinstance(indices, list) else [indices])}
        fog = StyleCSS("fog", fill=fill, stroke=stroke, stroke_width=1)
        ctx.builder.add_style(fog)
        for i in range(len(ctx.grid.hexes)):
            if i not in mapped:
                ctx.grid.hexes[i].style = fog
        return ""
    return OverlaySpec("fog", render, priority=6)


In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    
    

    ctx=ctx,
    debug = not showDemo
)

In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(fill="purple"),
        

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)


In [ ]:
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
sight = unit.place.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []

if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
unit.place.facing = HexPosition.SE
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
def SquadNamesOverlay(font_size: int = 14, font_family: str = "Cinzel", **kw) -> OverlaySpec:
    """Render squad names at the centroid of their placed pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            if not squad.name:
                continue
            
            # Collect fine-grid indices for all placed pieces
            placed_fine = []
            for piece in squad.pieces:
                if piece.place is None:
                    continue
                coarse_idx = piece.place.parent.hp2i(piece.place.location)
                for fi in ctx.fine_indices(coarse_idx):
                    if 0 <= fi < N:
                        placed_fine.append(fi)
            
            if not placed_fine:
                continue
            
            # Centroid of placed piece centers
            cx = sum(grid.hexes[fi].center.x for fi in placed_fine) / len(placed_fine)
            cy = sum(grid.hexes[fi].center.y for fi in placed_fine) / len(placed_fine)
            
            # Label style from flag
            label_style = squad.flag.labelStyle(f"sqname_{squad.name.replace(' ', '_')}")
            ctx.builder.add_style(label_style)
            ctx.builder.add_font(font_family)
            
            # Offset below centroid so it doesn't sit on top of pieces
            label_y = cy + grid.radius * 1.8
            
            parts.append(
                f'<text x="{cx:.1f}" y="{label_y:.1f}" text-anchor="middle" '
                f'font-size="{font_size}" font-family="\'{font_family}\', sans-serif" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{squad.name}</text>'
            )
        
        return '\n'.join(parts)
    
    return OverlaySpec("squad_names", render, requires={'squads'}, priority=90)


## Moves

In [ ]:
class MoveError(Exception):
    """Raised when a Move fails validation."""
    pass

In [ ]:
class Rotation(NamedTuple):
    piece: Piece
    facing: HexPosition
class Placement(NamedTuple):
    piece: Piece
    location: MapPlace
    
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None

    # Add occupied-hex check to Move validation
    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None, 
                placement: Placement | None = None) -> 'Move':
        rotations = rotations or []
        errors = []
        
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")
        
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")
        
        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")
        
        placed_allies = [p for p in squad.pieces if p.place is not None]
        if placement is not None and not placed_allies:
            if placement.piece.type != PieceType.KING:
                errors.append(f"{placement.piece.name}: first piece placed must be the King")
        
        # Sight check — use POST-rotation facing for rotated pieces
        if placement is not None and placement.piece in squad.pieces:
            if placed_allies:
                target_idx = placement.location.parent.hp2i(placement.location.location)
                
                # Build rotation lookup: piece -> new_facing
                rot_map = {r.piece: r.facing for r in rotations}
                
                visible = set()
                for ally in placed_allies:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    effective_sight = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        placement.location.parent.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, effective_sight)
                    )
                    visible |= sight_hexes
                
                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")
        
        # Check hex not already occupied
        if placement is not None and placement.location.parent is not None:
            target_idx = placement.location.parent.hp2i(placement.location.location)
            parts = placement.location.parent
            for sq in parts.squads:
                for p in sq.pieces:
                    if p.place is not None:
                        if parts.hp2i(p.place.location) == target_idx:
                            errors.append(f"{placement.piece.name}: hex {target_idx} already occupied by {p.name}")
        
        if errors:
            raise MoveError("; ".join(errors))
        
        return Move(rotations=rotations, placement=placement)


    #Move.from_squad = from_squad




In [ ]:
@patch
def apply(self: Move, parts: GameParts):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

In [ ]:
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    
    def record(self, squad: Squad, move: Move):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement)
        validated.apply(self.parts)
        turn = len(self._df)
        
        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=turn, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=turn, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame: return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0


@patch
def food_income(self: Squad, parts: GameParts) -> float:
    """Total food harvested by all placed pieces."""
    return sum(p.food.harvest * p.place.harvest
               for p in self.pieces if p.place is not None)

@patch
def food_upkeep(self: Squad) -> float:
    """Total diet cost for all placed pieces."""
    return sum(p.food.diet for p in self.pieces if p.place is not None)

@patch
def food_drain_suffered(self: Squad, all_squads: list['Squad']) -> float:
    """Total drain imposed on this squad by enemy sight cones."""
    my_placed = {p.place.parent.hp2i(p.place.location)
                 for p in self.pieces if p.place is not None}
    total = 0.0
    for enemy_squad in all_squads:
        if enemy_squad is self:
            continue
        for enemy in enemy_squad.pieces:
            if enemy.place is None:
                continue
            cone = enemy.sight().hexes
            caught = len(my_placed & cone)
            total += enemy.food.drain * caught
    return total

@patch
def food_net(self: Squad, parts: GameParts, all_squads: list['Squad']) -> float:
    """Net food change this turn (before placement costs)."""
    return self.food_income(parts) - self.food_upkeep() - self.food_drain_suffered(all_squads)


## Squad Network

In [ ]:
@patch
def to_nx(self: Squad) -> nx.DiGraph:
    """Build directed sight graph as NetworkX DiGraph with food attributes."""
    G = nx.DiGraph()
    placed = [p for p in self.pieces if p.place is not None]
    parts = placed[0].place.parent if placed else None
    if not parts:
        return G
    
    for p in placed:
        idx = parts.hp2i(p.place.location)
        G.add_node(p.id, 
                   piece=p,  # back-reference
                   piece_type=p.type.value,
                   harvest=int(p.food.harvest * p.place.harvest),
                   diet=p.food.diet,
                   drain=p.food.drain,
                   cost=p.food.cost,
                   pantry=p.place.pantry,
                   hex_idx=idx,
                   is_sink=p.food.harvest == 0.0 or p.type == PieceType.KING,
                   is_king=p.type == PieceType.KING)
    
    for p in placed:
        cone = p.sight().hexes
        for other in placed:
            if other is p:
                continue
            idx = parts.hp2i(other.place.location)
            if idx in cone:
                G.add_edge(p.id, other.id)
    
    return G


In [ ]:
@patch
def supply_bottlenecks(self: Squad) -> list[Piece]:
    """Pieces whose removal disconnects the sight network."""
    G = self.to_nx().to_undirected()
    return [G.nodes[n]['piece'] for n in nx.articulation_points(G)]

@patch
def isolated_pieces(self: Squad) -> list[Piece]:
    """Pieces not connected to the king's component."""
    G = self.to_nx().to_undirected()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return []
    king_component = nx.node_connected_component(G, king_id)
    return [G.nodes[n]['piece'] for n in G.nodes if n not in king_component]


In [ ]:
@patch
def food_sinks(self: Squad) -> list[Piece]:
    """Pieces that consume but don't produce — need to be fed by the network."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['is_sink']]

@patch
def food_sources(self: Squad) -> list[Piece]:
    """Pieces that can produce surplus (harvest > diet)."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['harvest'] > G.nodes[n]['diet']]

@patch
def flow_network(self: Squad) -> dict:
    """Analyse food flow using the sight graph.
    
    Returns dict with:
        G: the DiGraph
        sinks: set of sink node ids
        reachable: dict[node_id, set[sink_id]] — which sinks each piece can reach
        disconnected: list[Piece] — pieces that can't reach ANY sink
        sink_layers: dict[sink_id, dict[node_id, int]] — BFS distance from each sink
    """
    G = self.to_nx()
    if not G.nodes:
        return {'G': G, 'sinks': set(), 'reachable': {}, 
                'disconnected': [], 'sink_layers': {}}
    
    sinks = {n for n, d in G.nodes(data=True) if d['is_sink']}
    
    # For each sink, find all nodes that can reach it (reverse BFS)
    R = G.reverse()
    sink_connected = {}
    for sink in sinks:
        sink_connected[sink] = nx.ancestors(R, sink) | {sink}
    
    # Per-node: which sinks can it reach?
    reachable = {n: set() for n in G.nodes}
    for sink, connected in sink_connected.items():
        for n in connected:
            reachable[n].add(sink)
    
    disconnected = [G.nodes[n]['piece'] for n in G.nodes 
                    if not reachable[n] and n not in sinks]
    
    # BFS layers from each sink (along reverse edges = upstream distance)
    sink_layers = {}
    for sink in sinks:
        sink_layers[sink] = nx.shortest_path_length(R, sink)
    
    return {
        'G': G,
        'sinks': sinks,
        'sink_connected': sink_connected,
        'sink_layers': sink_layers,
        'reachable': reachable,
        'disconnected': disconnected,
    }




In [ ]:
@patch
def resolve_flow(self: Squad):
    """Apply food flow using nx max-flow through sight network."""
    G = self.to_nx()
    if len(G) < 2:
        return
    
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    
    # Build flow graph with super-source and super-sink
    F = nx.DiGraph()
    F.add_node('S')  # super-source: pieces with surplus
    F.add_node('T')  # super-sink: pieces in deficit
    
    for nid, data in G.nodes(data=True):
        pantry = data['piece'].place.pantry
        if pantry > 0:
            F.add_edge('S', nid, capacity=pantry)
        elif pantry < 0:
            F.add_edge(nid, 'T', capacity=-pantry)
    
    # Sight edges = relay capacity (effectively unlimited)
    for u, v in G.edges():
        F.add_edge(u, v, capacity=999)
        F.add_edge(v, u, capacity=999)  # sharing is bidirectional along sight
    
    if F.out_degree('S') == 0 or F.in_degree('T') == 0:
        return
    
    flow_val, flow_dict = nx.maximum_flow(F, 'S', 'T')
    
    # Apply: surplus pieces give, deficit pieces receive
    for nid in G.nodes:
        p = G.nodes[nid]['piece']
        donated = flow_dict.get('S', {}).get(nid, 0)
        received = flow_dict.get(nid, {}).get('T', 0)
        p.place.pantry -= donated   # gave from surplus
        p.place.pantry += received  # covered deficit
    
    # Remaining surplus → king (if connected)
    # Compute undirected view ONCE, no deepcopy
    if king_id:
        U = G.to_undirected(as_view=True)
        king_piece = G.nodes[king_id]['piece']
        for nid, data in G.nodes(data=True):
            if nid == king_id:
                continue
            p = data['piece']
            if p.place.pantry > 0 and nx.has_path(U, nid, king_id):
                king_piece.place.pantry += p.place.pantry
                p.place.pantry = 0


@patch
def starve(self: Squad, parts: GameParts, all_squads: list['Squad']) -> list[Piece]:
    RANK_ORDER = {PieceType.PAWN: 0, PieceType.BISHOP: 1, 
                  PieceType.KNIGHT: 1, PieceType.QUEEN: 2, PieceType.KING: 99}
    
    eliminated = []
    while self.food_net(parts, all_squads) < 0:
        G = self.to_nx()
        king_id = next((nid for nid, d in G.nodes(data=True) if d['is_king']), None)
        
        placed = [p for p in self.pieces 
                  if p.place is not None and p.type != PieceType.KING]
        if not placed: break
        
        # Graph distance to king (disconnected = infinity)
        if king_id:
            lengths = nx.shortest_path_length(G.to_undirected(), king_id)
        else:
            lengths = {}
        
        def starve_key(p):
            deficit = p.food.diet - (p.food.harvest * p.place.harvest)
            rank = RANK_ORDER.get(p.type, 0)
            graph_dist = lengths.get(p.id, 999)  # disconnected = first to go
            return (-deficit, rank, -graph_dist)
        
        victim = sorted(placed, key=starve_key)[0]
        victim.place = None
        eliminated.append(victim)
    
    return eliminated


@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)


In [ ]:
@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)

In [ ]:
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Cheap target-aware facing: blend ally connectivity with target projection."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    allies = {parts.hp2i(p.place.location) for p in placed}
    
    for piece in placed:
        my_hp = piece.place.location
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        best_dir, best_score = piece.place.facing, -999
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in my_hp.field_of_view(d, eff))
            
            # Connectivity: always valuable
            score = len((allies - {parts.hp2i(my_hp)}) & cone) * 3.0
            
            # King visibility bonus
            if king and king is not piece:
                if parts.hp2i(king.place.location) in cone:
                    score += 4.0
            
            # Target projection (the new part)
            if target_hp is not None:
                target_idx = parts.hp2i(target_hp)
                if target_idx in cone:
                    score += 8.0
                else:
                    # Partial: how far does cone extend toward target?
                    if cone:
                        min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                        score += 3.0 / max(1, min_dist)
            
            if score > best_score:
                best_score = score
                best_dir = d
        
        piece.place.facing = best_dir


@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Greedy coordinate descent over facings. Returns number of changes made."""
    return self.orient_facings(parts)
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return 0
    
    total_changes = 0
    for _ in range(max_iters):
        changed = False
        
        # Heuristic 3: optimize bottlenecks and king-adjacent pieces first
        G = self.to_nx()
        U = G.to_undirected()
        king_id = next((n for n,d in G.nodes(data=True) if d['is_king']), None)
        
        bottlenecks = set(nx.articulation_points(U)) if len(G) > 1 else set()
        king_neighbors = set(U.neighbors(king_id)) if king_id else set()
        
        def priority(p):
            nid = p.id
            is_bn = nid in bottlenecks
            is_kn = nid in king_neighbors
            return (not is_bn, not is_kn, p.food.harvest == 0)  # False sorts first
        
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            original = piece.place.facing
            best_facing = original
            best_score = _score_network(self, parts)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                s = _score_network(self, parts)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes


def _score_network(squad: Squad, parts: GameParts) -> float:
    """Score a squad's current facing configuration."""
    G = squad.to_nx()
    if len(G) < 2: return 0.0
    
    king_id = next((n for n,d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return 0.0
    U = G.to_undirected()
    
    # 1. Connectivity — most important
    king_comp = nx.node_connected_component(U, king_id)
    n_connected = len(king_comp)
    n_total = len(G)
    connectivity = (n_connected / n_total) * 100
    
    # 2. Total harvest reachable to king via the network
    food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
    
    # 3. Enemy drain coverage
    drain_score = 0.0
    for n in G.nodes:
        p = G.nodes[n]['piece']
        if p.food.drain > 0:
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is not None:
                        if parts.hp2i(ep.place.location) in cone:
                            drain_score += p.food.drain * 0.1
    
    # 4. Penalize bottlenecks (fragile network)
    bottleneck_penalty = len(list(nx.articulation_points(U))) * 5
    
    # 5. Bonus for redundant sight links (robustness)
    edge_bonus = U.number_of_edges() * 0.5
    
    return connectivity + food_reach + drain_score - bottleneck_penalty + edge_bonus


In [ ]:
@patch
def tick_lifespan(self: Squad) -> list[Piece]:
    """Increment tenure for placed pieces. Evict any that exceed lifespan."""
    evicted = []
    for p in self.pieces:
        if p.place is None or p.type == PieceType.KING:
            continue
        p.place.tenure += 1
        if p.place.tenure >= LIFESPAN.get(p.type, 4):
            p.place = None
            evicted.append(p)
    return evicted


@patch
def tick_food(self: Squad, parts: GameParts, all_squads: list['Squad']):
    """Per-piece food resolution with graph-based sharing."""
    placed = [p for p in self.pieces if p.place is not None]
    
    # 1. Harvest + pay diet
    for p in placed:
        p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
    
    # 2. Sharing via max-flow through sight network
    G = self.to_nx()
    if len(G) > 1:
        F = nx.DiGraph()
        F.add_node('S')  # super-source
        F.add_node('T')  # super-sink
        for nid, data in G.nodes(data=True):
            pantry = data['piece'].place.pantry
            if pantry > 0:
                F.add_edge('S', nid, capacity=pantry)
            elif pantry < 0:
                F.add_edge(nid, 'T', capacity=-pantry)
        # Sight edges = unlimited sharing capacity
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)  # sharing is bidirectional
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            flow_val, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for nid in G.nodes:
                sent = sum(v for k, v in flow_dict.get(nid, {}).items() if k != 'S' and k != 'T')
                received = sum(flow_dict.get(other, {}).get(nid, 0) 
                              for other in G.nodes if other != nid)
                G.nodes[nid]['piece'].place.pantry += (received - sent)
    
    # 3. Enemy drain
    for p in placed:
        cone = p.sight().hexes
        for enemy_squad in all_squads:
            if enemy_squad is self: continue
            for enemy in enemy_squad.pieces:
                if enemy.place is None: continue
                if parts.hp2i(enemy.place.location) in cone:
                    enemy.place.pantry -= int(p.food.drain)


In [ ]:
def FoodNetworkOverlay(
    edge_opacity=0.6, edge_dash="6,4",
    bottleneck_r_mult=1.4,
    **kw
) -> OverlaySpec:
    """Food sharing network using flag-themed glyphs and MapPaths."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            G = squad.to_nx()
            if len(G) < 2:
                continue
            
            flag = squad.flag
            glyphs = DiagramGlyphs(flag, size=grid.radius * 0.5)
            glyphs.register_styles(ctx.builder)
            
            # Edge style from flag colors
            edge_style = StyleCSS(
                f"foodnet_edge_{flag.name}",
                fill="none", stroke=flag.darkPrimary,
                stroke_width=max(1.5, grid.radius * 0.06),
                stroke_linecap="round", opacity=edge_opacity)
            ctx.builder.add_style(edge_style)
            
            # Surplus/deficit styles
            surplus_style = StyleCSS(
                f"foodnet_surplus_{flag.name}",
                fill=flag.primary, stroke=flag.darkPrimary,
                stroke_width=1.5, opacity=0.85)
            deficit_style = StyleCSS(
                f"foodnet_deficit_{flag.name}",
                fill=flag.comp, stroke=flag.darkPrimary,
                stroke_width=1.5, opacity=0.85)
            ctx.builder.add_style(surplus_style)
            ctx.builder.add_style(deficit_style)
            
            # Bottleneck detection
            bottlenecks = set(nx.articulation_points(G.to_undirected()))
            
            # --- Edges as MapPath arrows ---
            seen_edges = set()
            for u, v in G.to_undirected().edges():
                key = tuple(sorted([u, v]))
                if key in seen_edges:
                    continue
                seen_edges.add(key)
                
                u_idx = G.nodes[u]['hex_idx']
                v_idx = G.nodes[v]['hex_idx']
                
                for fi_u in ctx.fine_indices(u_idx):
                    for fi_v in ctx.fine_indices(v_idx):
                        if 0 <= fi_u < N and 0 <= fi_v < N:
                            c1 = grid.hexes[fi_u].center
                            c2 = grid.hexes[fi_v].center
                            
                            # Shorten endpoints toward center (don't overlap nodes)
                            shrink = grid.radius * 0.4
                            dx = c2.x - c1.x
                            dy = c2.y - c1.y
                            dist = math.sqrt(dx*dx + dy*dy)
                            if dist > 0:
                                ux, uy = dx/dist, dy/dist
                                p1 = MapCord(c1.x + ux*shrink, c1.y + uy*shrink)
                                p2 = MapCord(c2.x - ux*shrink, c2.y - uy*shrink)
                            else:
                                p1, p2 = c1, c2
                            
                            path = MapPath([p1, p2], edge_style)
                            parts.append(
                                path.drawPolygon(f'stroke-dasharray="{edge_dash}"'))
            
            # --- Nodes ---
            for nid, data in G.nodes(data=True):
                piece = data['piece']
                harvest = data['harvest']
                diet = data['diet']
                net = harvest - diet
                idx = data['hex_idx']
                is_bottleneck = nid in bottlenecks
                
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        c = grid.hexes[fi].center
                        r = grid.radius * 0.25
                        
                        # Bottleneck: warning ring
                        if is_bottleneck:
                            parts.append(glyphs.ring(
                                c, r * bottleneck_r_mult,
                                glyphs.danger_style()))
                        
                        # Node glyph based on role
                        if net > 0:
                            # Harvester: sheaf icon
                            parts.append(glyphs.sheaf(c, size=r * 2.5))
                        elif net < 0:
                            # Consumer: shield icon
                            parts.append(glyphs.shield(c, size=r * 2))
                        else:
                            # Break-even: simple ring
                            parts.append(glyphs.ring(c, r))
                        
                        # Food pip (red-green gradient circle)
                        pip_frac = (net + 5) / 10.0  # normalize roughly
                        pip_center = MapCord(c.x + r * 1.8, c.y - r * 1.8)
                        parts.append(glyphs.food_pip(
                            pip_center, pip_frac,
                            max_r=r * 0.8, min_r=r * 0.3))
                        
                        # Net label
                        label = f"{net:+d}" if net != 0 else "0"
                        parts.append(
                            f'<text x="{c.x:.1f}" y="{c.y + r * 2.5:.1f}" '
                            f'text-anchor="middle" font-size="{max(9, grid.radius * 0.3):.0f}" '
                            f'fill="{flag.darkPrimary}" font-weight="bold" '
                            f'opacity="0.8">{label}</text>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)


In [ ]:
if showDemo:
    ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        FoodNetworkOverlay(),
        pieces=myStuff.squads[0].pieces,
        ctx=ctx,
        debug = not showDemo
    )


## TurnProcessor

In [ ]:
@dataclass
class TurnProcessor:
    parts: GameParts
    squads: list[Squad]
    log: MoveLog = None
    turn: int = 0
    game_over: bool = False
    winner: Squad | None = None
    
    def __post_init__(self):
        if self.log is None:
            self.log = MoveLog(self.parts)
    
    def _king(self, squad: Squad) -> Piece | None:
        return next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    
    def play_round(self, moves: dict[Squad, Move]) -> list[dict]:
        summaries = []
        for squad in self.squads:
            if squad not in moves or not squad.king_alive():
                continue
            s = self.resolve_move(squad, moves[squad])
            summaries.append(s)
            if self.game_over:
                break
        self.turn += 1
        return summaries
    
    def _check_game_over(self):
        alive = [s for s in self.squads if s.king_alive()]
        if len(alive) <= 1:
            self.game_over = True
            self.winner = alive[0] if alive else None
    
    def status(self) -> pd.DataFrame:
        rows = []
        for s in self.squads:
            king = self._king(s)
            placed = [p for p in s.pieces if p.place is not None]
            G = s.to_nx()
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            bottlenecks = len(list(nx.articulation_points(G.to_undirected()))) if len(G) > 1 else 0
            rows.append({
                'squad': s.name,
                'king_pantry': king.place.pantry if king else 0,
                'placed': len(placed),
                'bottlenecks': bottlenecks,
                'king': '👑' if s.king_alive() else '💀',
            })
        return pd.DataFrame(rows)


In [ ]:
@patch
def resolve_move(self:TurnProcessor, squad: Squad, move: Move) -> dict:
    if self.game_over:
        raise MoveError("Game is over")
    
    king = self._king(squad)
    
    summary = {
        'squad': squad.name, 'turn': self.turn,
        'placement_cost': 0, 'placement_pantry': 0,
        'food_harvested': 0.0, 'food_diet': 0.0,
        'food_drain_dealt': 0.0,
        'starved': [], 'expired': [],
        'connected': 0, 'disconnected': 0,
        'king_pantry_before': king.place.pantry if king else 0,
        'king_pantry_after': 0,
        'pantries': {},
        'game_over': False, 'eliminated': None,
    }
    
    # 1. Validate & apply
    self.log.record(squad, move)
    
    # 2. Placement cost from king, seed pantry
    if move.placement is not None:
        piece = move.placement.piece
        cost = piece.food.cost
        king = self._king(squad)
        if king and piece.type != PieceType.KING:
            king.place.pantry -= cost
        piece.place.pantry = int(cost * 0.75)
        summary['placement_cost'] = cost
        summary['placement_pantry'] = piece.place.pantry
    
    # 3. Tick lifespans
    expired = squad.tick_lifespan()
    summary['expired'] = [p.name for p in expired]
    
    king = self._king(squad)
    placed = [p for p in squad.pieces if p.place is not None]
    
    # 4a. Harvest
    for p in placed:
        earned = int(p.food.harvest * p.place.harvest)
        p.place.pantry += earned
        summary['food_harvested'] += earned
    
    # 4b. Diet
    for p in placed:
        p.place.pantry -= p.food.diet
        summary['food_diet'] += p.food.diet
    
    # 4c. Enemy drain
    for p in placed:
        cone = p.sight().hexes
        for enemy_sq in self.squads:
            if enemy_sq is squad:
                continue
            for enemy in enemy_sq.pieces:
                if enemy.place is None:
                    continue
                if self.parts.hp2i(enemy.place.location) in cone:
                    drain = int(p.food.drain)
                    enemy.place.pantry -= drain
                    summary['food_drain_dealt'] += drain
    
    # 5. Network flow — the nx magic
    G = squad.to_nx()
    squad.resolve_flow()
    
    # Track connectivity
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if king_id:
        U = G.to_undirected()
        connected = sum(1 for n in G.nodes if nx.has_path(U, n, king_id))
        summary['connected'] = connected
        summary['disconnected'] = len(G) - connected
    
    # 6. Starve pieces still in deficit
    RANK_ORDER = {
        PieceType.PAWN: 0, PieceType.KNIGHT: 1,
        PieceType.BISHOP: 2, PieceType.QUEEN: 3, PieceType.KING: 99,
    }
    king_hp = king.place.location if king else None
    starving = [p for p in placed 
                if p.place is not None and p.place.pantry < 0 
                and p.type != PieceType.KING]
    starving.sort(key=lambda p: (
        p.place.pantry,
        RANK_ORDER.get(p.type, 0),
        -(p.place.location.distance(king_hp) if king_hp else 0)
    ))
    for p in starving:
        p.place = None
        summary['starved'].append(p.name)
    
    # 7. Snapshot
    summary['pantries'] = {
        p.name: p.place.pantry for p in squad.pieces if p.place is not None
    }
    summary['king_pantry_after'] = king.place.pantry if king else 0
    
    
    # 8. Win condition
    if not squad.king_alive():
        summary['game_over'] = True
        summary['eliminated'] = squad.name
        self._check_game_over()
    
    # 8b. King death — bankrupt
    king = self._king(squad)
    if king and king.place.pantry <= 0:
        king.place = None
        summary['game_over'] = True
        summary['eliminated'] = squad.name
        self._check_game_over()

    return summary




Which functions should i remove from squads since resolve_move

Did refactor correctly?

In [ ]:
@patch(as_prop=True)
def land_components(self: GameParts) -> dict[int, int]:
    """Map each land hex to its connected component ID. Computed once."""
    if hasattr(self, '_land_comp'):
        return self._land_comp
    
    comp = {}
    cid = 0
    for i in range(len(self.terr.elevations)):
        if self.terr.elevations[i] <= 0 or i in comp:
            continue
        frontier = {i}
        while frontier:
            cur = frontier.pop()
            if cur in comp:
                continue
            comp[cur] = cid
            for nb in self.grid.neighborsOf(cur):
                if nb not in comp and self.terr.elevations[nb] > 0:
                    frontier.add(nb)
        cid += 1
    self._land_comp = comp
    return comp

@patch
def land_connected(self: GameParts, indices: list[int]) -> bool:
    comp = self.land_components
    cids = {comp.get(i) for i in indices}
    return len(cids) == 1 and None not in cids


In [ ]:
@patch
def find_starts(self: GameParts, n: int, ring_radius: int = 3, 
                min_distance: int = 6, balance_tolerance: float = 0.3,
                top_k: int = 40) -> list[int]:
    grid, terr, tiers = self.grid, self.terr, self.fy.tiers
    n_hexes = len(terr.elevations)
    
    # 1. Richness scores
    scores = np.zeros(n_hexes)
    for i in range(n_hexes):
        if terr.elevations[i] <= 0: continue
        neighbors = grid.indices_in_range(i, ring_radius)
        scores[i] = sum(tiers[j] for j in neighbors if terr.elevations[j] > 0)
    
    # 2. Top candidates sorted by score
    ranked = sorted([(i, scores[i]) for i in range(n_hexes) if scores[i] > 0],
                    key=lambda x: -x[1])[:top_k]
    
    # 3. Precompute hex distances between candidates
    def hex_dist(a, b):
        return grid.index_to_hexposition(a).distance(grid.index_to_hexposition(b))
    
    # 4. Try combinations, best balanced first
    best_set, best_spread = None, float('inf')
    
    for combo in combinations(range(len(ranked)), n):
        idxs = [ranked[c][0] for c in combo]
        scs = [ranked[c][1] for c in combo]
        
        # Distance check
        if any(hex_dist(a, b) < min_distance 
               for a, b in combinations(idxs, 2)):
            continue
        
        # Connectivity check
        if not self.land_connected(idxs):
            continue
        
        # Balance check
        median = np.median(scs)
        if median == 0: continue
        spread = (max(scs) - min(scs)) / median
        
        if spread < best_spread:
            best_spread = spread
            best_set = list(zip(idxs, scs))
            if spread <= balance_tolerance:
                break
    
    if best_set is None:
        raise ValueError(f"Could not find {n} balanced land-connected positions")
    
    result = [idx for idx, _ in best_set]
    print(f"Start positions (spread={best_spread:.2f}):")
    for idx, score in best_set:
        print(f"  Hex {idx}: richness={score:.0f}, level={terr.elevationLevel(idx)}, tier={tiers[idx]}")
    
    return result


In [ ]:
playerCount = 5
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads( year=1985, seed = 23,levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
# Place each squad's king
for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)

# Check it worked
if showDemo:
    for sq in myStuff.squads:
        show(sq.table())

## Targets

In [ ]:
def _score_network(squad: Squad, parts: GameParts) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    G = squad.to_nx()
    if len(G) < 2: return 0.0
    
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return 0.0
    U = G.to_undirected()
    
    # --- Base: connectivity (always matters) ---
    king_comp = nx.node_connected_component(U, king_id)
    n_connected = len(king_comp)
    n_total = len(G)
    connectivity = (n_connected / n_total) * 100
    bottleneck_penalty = len(list(nx.articulation_points(U))) * 5
    edge_bonus = U.number_of_edges() * 0.5
    
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    # --- CONSOLIDATE or no target: old logic ---
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    
    if target.type == TargetType.KING:
        # --- KING target: maximize drain on enemies near target ---
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    ep_idx = parts.hp2i(ep.place.location)
                    if ep_idx in cone:
                        # Weight by proximity to target king
                        ep_hp = ep.place.location
                        dist_to_target = max(1, ep_hp.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        # Reward sight cones that reach toward the target
        projection = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            cone = p.sight().hexes
            if target.hex_idx in cone:
                projection += 10.0
            else:
                # Partial credit: closest cone hex to target
                if cone:
                    min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                    projection += 2.0 / max(1, min_dist)
        
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        # --- OASIS target: maximize new land revealed toward target ---
        land_set = {i for i in range(len(parts.terr.elevations)) 
                    if parts.terr.elevations[i] > 0}
        
        # Collective sight
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        
        new_land = len(all_visible & land_set)
        
        # Reward sight that extends toward the oasis
        projection = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            cone = p.sight().hexes
            if target.hex_idx in cone:
                projection += 10.0
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        projection += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        
        return base + new_land * 1.5 + projection * 2.0 + food_reach


In [ ]:


@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Coordinate descent over facings using target-aware objective.
    
    Expensive — call when setting a new 5-turn plan, not every turn.
    Falls back to orient_facings for priority ordering of first pass.
    """
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2:
        return 0
    
    # Seed with cheap heuristic first
    self.orient_facings(parts)
    
    # Priority: optimize bottlenecks and frontier pieces first
    G = self.to_nx()
    U = G.to_undirected()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    bottlenecks = set(nx.articulation_points(U)) if len(G) > 1 else set()
    king_neighbors = set(U.neighbors(king_id)) if king_id else set()
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    def priority(p):
        nid = p.id
        is_bn = nid in bottlenecks
        is_kn = nid in king_neighbors
        # Frontier = closest to target (optimize these first for KING/OASIS)
        if target_hp:
            dist = p.place.location.distance(target_hp)
        else:
            dist = 999
        return (not is_bn, not is_kn, dist)
    
    total_changes = 0
    
    for _ in range(max_iters):
        changed = False
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            if piece.place is None: continue
            original = piece.place.facing
            best_facing = original
            best_score = _score_network(self, parts)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                s = _score_network(self, parts)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes


In [ ]:
class TargetType(Enum):
    KING = "king"
    OASIS = "oasis"
    CONSOLIDATE = "consolidate"


In [ ]:
@dataclass
class SquadTarget:
    type: TargetType
    hex_idx: int = -1         # -1 for CONSOLIDATE
    value: float = 0.0
    enemy_squad: str = ""
    turns_remaining: int = 5  # countdown within strategy cycle
    
    @property
    def label(self) -> str:
        if self.type == TargetType.CONSOLIDATE:
            return f"🏰 consolidate ({self.turns_remaining}t left)"
        if self.type == TargetType.KING:
            return f"⚔️ {self.enemy_squad} king @ {self.hex_idx} ({self.turns_remaining}t)"
        return f"🌾 oasis @ {self.hex_idx} ({self.turns_remaining}t)"
    
    @property
    def is_expired(self) -> bool:
        return self.turns_remaining <= 0
    
    def tick(self):
        self.turns_remaining = max(0, self.turns_remaining - 1)


In [ ]:
@patch
def choose_target(self: Squad, parts: GameParts, 
                  strategy: PlacementStrategy,
                  oasis_radius: int = 3) -> SquadTarget:
    """Pick target based on strategy personality + board state."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return SquadTarget(TargetType.CONSOLIDATE)
    
    stance = strategy.assess(self, parts)
    my_hp = king.place.location
    
    # --- Strategy × stance → preferred target type ---
    PREFERENCE = {
        "aggressive": {"expand": "king",  "defend": "king",        "attack": "king"},
        "expansive":  {"expand": "oasis", "defend": "consolidate", "attack": "oasis"},
        "balanced":   {"expand": "oasis", "defend": "consolidate", "attack": "king"},
        "turtle":     {"expand": "consolidate", "defend": "consolidate", "attack": "oasis"},
    }
    preferred = PREFERENCE.get(strategy.name, PREFERENCE["balanced"])[stance]
    
    # --- Score candidate kings ---
    king_candidates = []
    for sq in parts.squads:
        if sq is self: continue
        eking = next((p for p in sq.pieces 
                      if p.type == PieceType.KING and p.place is not None), None)
        if not eking: continue
        eidx = parts.hp2i(eking.place.location)
        dist = my_hp.distance(eking.place.location)
        if dist == 0: continue
        placed = sum(1 for p in sq.pieces if p.place is not None)
        weakness = 1.0 / max(1, eking.place.pantry) + 1.0 / max(1, placed)
        score = weakness / dist
        king_candidates.append(SquadTarget(
            TargetType.KING, eidx, value=score, enemy_squad=sq.name))
    
    # --- Score candidate oases ---
    oasis_candidates = []
    tiers = parts.fy.tiers
    terr = parts.terr
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    for i in range(len(terr.elevations)):
        if terr.elevations[i] <= 0: continue
        ring = parts.grid.indices_in_range(i, oasis_radius)
        richness = sum(tiers[j] for j in ring 
                       if terr.elevations[j] > 0 and j not in occupied)
        if richness < 15: continue
        dist = my_hp.distance(parts.i2hp(i))
        if dist == 0: continue
        score = richness / dist
        oasis_candidates.append(SquadTarget(TargetType.OASIS, i, value=score))
    
    # --- Pick based on preference, with fallback ---
    if preferred == "king" and king_candidates:
        return max(king_candidates, key=lambda t: t.value)
    if preferred == "oasis" and oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    if preferred == "consolidate":
        return SquadTarget(TargetType.CONSOLIDATE)
    
    # Fallback chain: oasis → consolidate
    if oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    return SquadTarget(TargetType.CONSOLIDATE)


## Best Moves

In [ ]:
@patch
def best_pawn_move(self: Squad, parts: GameParts) -> Move | None:
    """Find the best pawn placement: highest yield hex in or near ally sight.
    
    Returns a Move with optional rotation + pawn placement, or None if no pawns available.
    """
    # Find an unplaced pawn
    pawn = next((p for p in self.pieces 
                 if p.type == PieceType.PAWN and p.place is None), None)
    if pawn is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    grid = parts.grid
    terr = parts.terr
    tiers = parts.fy.tiers
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # First: check all hexes visible from current sight (no rotation needed)
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Filter to valid land hexes not already occupied
    candidates = {idx for idx in current_visible 
                  if 0 <= idx < len(terr.elevations) 
                  and terr.elevations[idx] > 0 
                  and idx not in occupied}
    
    best_idx = None
    best_yield = -1
    best_rotation = None  # (piece, new_facing) if rotation needed
    
    # Check current sight first
    for idx in candidates:
        if tiers[idx] > best_yield:
            best_yield = tiers[idx]
            best_idx = idx
            best_rotation = None
    
    # Now try each piece × each facing to find even better options
    dirs = HexPosition.directions()
    for ally in placed:
        original_facing = ally.place.facing
        for new_facing in dirs:
            if new_facing == original_facing:
                continue
            # Temporarily compute sight with new facing
            bonus = ally.food.bonus_sight
            effective_sight = max(2, ally.place.level) + bonus
            new_sight = set(
                parts.hp2i(hp) 
                for hp in ally.place.location.field_of_view(new_facing, effective_sight)
            )
            for idx in new_sight:
                if (0 <= idx < len(terr.elevations) 
                    and terr.elevations[idx] > 0 
                    and idx not in occupied
                    and tiers[idx] > best_yield):
                    best_yield = tiers[idx]
                    best_idx = idx
                    best_rotation = Rotation(ally, new_facing)
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations, 
                           placement=Placement(pawn, loc))


In [ ]:
@patch
def best_queen_move(self: Squad, parts: GameParts, 
                    stance: str = "defend") -> Move | None:
    """Place a queen strategically based on stance.
    
    stance='defend': prioritize king protection + ally coverage
    stance='attack': prioritize draining enemy pieces
    """
    queen = next((p for p in self.pieces 
                  if p.type == PieceType.QUEEN and p.place is None), None)
    if queen is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < queen.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Ally + enemy positions
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    enemy_idxs = set()
    for sq in parts.squads:
        if sq is self: continue
        for p in sq.pieces:
            if p.place is not None:
                enemy_idxs.add(parts.hp2i(p.place.location))
    
    king_hp = king.place.location if king else None
    
    # All valid placement hexes
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    candidates = {idx for idx in current_visible
                  if 0 <= idx < len(terr.elevations)
                  and terr.elevations[idx] > 0
                  and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = queen.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        
        # Distance to king (lower = better for defense)
        king_dist = hp.distance(king_hp) if king_hp else 0
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            caught_enemies = len(enemy_idxs & cone)
            allies_covered = len(ally_idxs & cone)
            
            if stance == "defend":
                # Fortress: protect the king, cover allies, still punish nearby enemies
                score = (
                    allies_covered * 3.0
                    - king_dist * 4.0
                    + caught_enemies * queen.food.drain * 1.0
                    + (5.0 if king_hp and parts.hp2i(king_hp) in cone else 0)  # bonus: can see king
                )
            else:  # "attack" — you miss 100% of the shots you don't take
                score = (
                    caught_enemies * queen.food.drain * 3.0
                    + allies_covered * 1.0
                    - king_dist * 1.0  # still prefer not to be stranded
                )
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't currently visible
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(queen, loc))


In [ ]:
@patch
def best_bishop_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a bishop using auto-detected role: relay, harvester, or frontier."""
    bishop = next((p for p in self.pieces 
                   if p.type == PieceType.BISHOP and p.place is None), None)
    if bishop is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < bishop.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Land hex set for frontier scoring
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    # --- Auto-detect role weights ---
    G = self.to_nx()
    U = G.to_undirected() if len(G) > 1 else nx.Graph()
    has_bottleneck = len(list(nx.articulation_points(U))) > 0 if len(G) > 1 else False
    isolated = self.isolated_pieces()
    king_pantry = king.place.pantry if king else 0
    food_tight = king_pantry < 30
    
    if has_bottleneck or len(isolated) > 0:
        # Network fragile — bridge it
        w_connect, w_drain, w_harvest, w_frontier = 15, 1, 1, 3
        role = "relay"
    elif food_tight:
        # Running low — earn food
        w_connect, w_drain, w_harvest, w_frontier = 5, 1, 8, 1
        role = "harvester"
    else:
        # Healthy — expand vision for future placements
        w_connect, w_drain, w_harvest, w_frontier = 5, 3, 2, 6
        role = "frontier"
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = bishop.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        tier = int(parts.fy.tiers[idx])
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            
            # Connectivity: allies this bishop can see
            allies_seen = len(ally_idxs & cone)
            
            # Drain: enemies caught
            enemies_caught = len(enemy_idxs & cone)
            
            # Harvest: net food over 4-turn lifespan
            harvest_per_turn = bishop.food.harvest * tier - bishop.food.diet
            lifetime_net = harvest_per_turn * LIFESPAN[PieceType.BISHOP] - bishop.food.cost
            
            # Frontier: new land hexes revealed that nobody currently sees
            new_land = len((cone & land_set) - current_visible)
            
            score = (allies_seen * w_connect
                     + enemies_caught * bishop.food.drain * w_drain
                     + lifetime_net * w_harvest
                     + new_land * w_frontier
                     + loc.level * 0.5)  # slight high-ground tiebreak
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't in current sight
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(bishop, loc))


In [ ]:
@patch
def best_knight_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a knight as a one-turn drain missile targeting enemy clusters."""
    knight = next((p for p in self.pieces 
                   if p.type == PieceType.KNIGHT and p.place is None), None)
    if knight is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < knight.food.cost + 5:
        return None  # don't fire blanks when we're broke
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Build enemy position map with piece value for targeting priority
    PIECE_VALUE = {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.BISHOP: 4, PieceType.PAWN: 2, PieceType.KNIGHT: 1,
    }
    enemy_map = {}  # idx -> total value of enemies at that hex
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is not None:
                idx = parts.hp2i(p.place.location)
                enemy_map[idx] = enemy_map.get(idx, 0) + PIECE_VALUE.get(p.type, 1)
    
    if not enemy_map:
        return None  # no targets, save the food
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    best_rotation = None
    
    # Also try with one rotation to unlock new hexes
    sight_configs = [(None, current_visible)]  # (rotation, visible_set)
    for ally in placed:
        original = ally.place.facing
        b = ally.food.bonus_sight
        eff = max(2, ally.place.level) + b
        for d in dirs:
            if d == original:
                continue
            new_cone = set(
                parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
            # Recompute collective sight with this rotation
            expanded = (current_visible - ally.sight().hexes) | new_cone
            sight_configs.append((Rotation(ally, d), expanded))
    
    for rotation, visible in sight_configs:
        cands = {idx for idx in visible
                 if idx in land_set and idx not in occupied}
        
        for idx in cands:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            eff_sight = max(2, loc.level)  # knight has 0 bonus_sight
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Total drain value: enemies caught weighted by importance
                drain_value = sum(enemy_map.get(eidx, 0) 
                                  for eidx in (cone & set(enemy_map.keys())))
                enemies_hit = len(cone & set(enemy_map.keys()))
                
                # ROI: is draining worth the 6 food cost?
                total_drain = enemies_hit * knight.food.drain
                roi = total_drain - knight.food.cost - knight.food.diet
                
                if roi <= 0:
                    continue  # not worth deploying
                
                score = (drain_value * knight.food.drain * 2.0
                         + roi * 1.5
                         + loc.level * 0.3)  # high ground = bigger cone
                
                if score > best_score:
                    best_score = score
                    best_idx = idx
                    best_facing = d
                    best_rotation = rotation
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(knight, loc))


In [ ]:
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Pawn round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            idx = myStuff.hp2i(move.placement.location.location) if move.placement else "?"
            print(f"  {squad.name}: → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")

print(f"\n{tp.status()}")


In [ ]:
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Pawn round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            idx = myStuff.hp2i(move.placement.location.location) if move.placement else "?"
            print(f"  {squad.name}: → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
    
    # Optimize facings after each round of placements
    print(f"  --- Optimizing facings ---")
    for squad in myStuff.squads:
        changes = squad.optimize_facings(myStuff)
        if changes:
            print(f"  {squad.name}: {changes} facing(s) adjusted")

print(f"\n{tp.status()}")


In [ ]:
# Module-level scorer functions (shared by overlay + panel)
def _score_pawn(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    return int(parts.fy.tiers[idx])

def _score_bishop(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.BISHOP].bonus_sight
    eff = max(2, loc.level) + bonus
    tier = int(parts.fy.tiers[idx])
    dirs = HexPosition.directions()
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_seen = len(ally_idxs & cone)
        new_land = len((cone & land_set) - current_visible)
        harvest_net = FOOD_TABLE[PieceType.BISHOP].harvest * tier - FOOD_TABLE[PieceType.BISHOP].diet
        score = allies_seen * 5 + new_land * 4 + harvest_net * 3 + loc.level * 0.5
        best = max(best, score)
    return best

def _score_knight(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    eff = max(2, loc.level)
    dirs = HexPosition.directions()
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    if not enemy_idxs: return -999
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        enemies_hit = len(cone & enemy_idxs)
        roi = enemies_hit * FOOD_TABLE[PieceType.KNIGHT].drain - FOOD_TABLE[PieceType.KNIGHT].cost
        if roi <= 0: continue
        best = max(best, roi + loc.level * 0.3)
    return best

def _score_queen(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.QUEEN].bonus_sight
    eff = max(2, loc.level) + bonus
    dirs = HexPosition.directions()
    king = next((p for p in squad.pieces if p.type == PieceType.KING and p.place is not None), None)
    king_hp = king.place.location if king else None
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    king_dist = hp.distance(king_hp) if king_hp else 0
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_covered = len(ally_idxs & cone)
        enemies_caught = len(enemy_idxs & cone)
        sees_king = 1 if king_hp and parts.hp2i(king_hp) in cone else 0
        score = (allies_covered * 3.0 - king_dist * 4.0
                 + enemies_caught * FOOD_TABLE[PieceType.QUEEN].drain
                 + sees_king * 5.0)
        best = max(best, score)
    return best

PLACEMENT_SCORERS = {
    PieceType.PAWN:   _score_pawn,
    PieceType.BISHOP: _score_bishop,
    PieceType.KNIGHT: _score_knight,
    PieceType.QUEEN:  _score_queen,
}


In [ ]:
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag
            
            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}
            
            for pt, scorer in PLACEMENT_SCORERS.items():
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced: continue
                
                scored = []
                for idx in current_visible:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s > -999:
                        scored.append((idx, s))
                if not scored: continue
                
                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]
                
                scale = grid.radius * 0.7 / 22.5
                
                for rank, (idx, score) in enumerate(top):
                    opacity = 0.8 - rank * 0.2
                    
                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            
                            # Ghost ring background
                            r = grid.radius * 0.6
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{opacity * 0.5:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{opacity:.2f}" '
                                f'stroke-width="1.5" stroke-dasharray="3,2"/>')
                            
                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{opacity:.2f}">{svg_str}</g>')
                            
                            # Rank number (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{opacity:.2f}">{rank + 1}</text>')
                            
                            # Hex index (bottom)
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{opacity * 0.7:.2f}">#{idx}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)


# HexMagic Game — Architecture Reference

## Overview

A hex-based strategy game where squads of chess-typed pieces compete for territory on a procedurally generated terrain map. The core mechanic is **food economics** — pieces harvest food from tiles, share it along directed sight networks, and drain enemy resources. The king's pantry is the squad treasury; lose the king and you lose the game.

---

## Core Data Structures

### GameParts
The central game state container.
- `terr: TerraDemo` — procedural terrain (elevations, climate, precipitation)
- `grid: HexGrid` — hex layout with coordinate conversions
- `basins: DrainageBasins` — river/watershed data
- `fy: FoodYield` — per-hex food tier (0–7), computed from temperature × water × soil
- `squads: list[Squad]` — all players
- `flags: list[CountryFlag]` — visual identity per squad
- `referenceIndex: int` — grid center, used for `HexPosition ↔ index` conversion

Key methods:
- `hp2i(HexPosition) → int` — hex position to grid index
- `i2hp(int) → HexPosition` — grid index to hex position
- `location_by_index(int) → MapPlace` — creates a MapPlace with elevation level and food tier
- `find_starts(n, ring_radius, min_distance, balance_tolerance)` — finds n balanced king start positions
- `land_components` (property) — connected component IDs for land hexes
- `land_connected(indices)` — checks all indices share one land component
- `overlayContext(region, padding, radius)` — builds a GameContext for SVG rendering

### MapPlace
Per-hex placement info for a piece.
- `location: HexPosition` — cube coordinates
- `facing: HexPosition` — one of 6 hex directions (W, E, NW, NE, SW, SE)
- `level: int` — elevation tier (used for sight range)
- `harvest: int` — food yield tier of this tile
- `parent: GameParts` — back-reference
- `tenure: int` — turns spent at this location
- `pantry: int` — per-piece food storage (the individual ledger)

### Piece
An individual game unit.
- `id: str` — unique 16-char random string
- `flag: CountryFlag` — visual identity
- `type: PieceType` — PAWN, BISHOP, KNIGHT, QUEEN, or KING
- `place: MapPlace | None` — None if not on board
- `name: str` — generated name (era-appropriate)
- `year: int` — birth year
- `lantern: StyleCSS` — sight cone visual style

Key methods:
- `food` (property) → `FoodStats` — diet, cost, harvest multiplier, drain, bonus_sight
- `sight() → HexRegion` — hex indices in this piece's field of view (facing + level + bonus)
- `visible_allies(squad) → list[Piece]` — allies in sight cone, sorted oldest-first then alphabetically

### FoodStats (NamedTuple)

## Piece Strategy Reference

### ♟ Pawn — "Settler"
**Profile:** diet=2, cost=7, harvest=1.25, drain=0.25, sight=base, lifespan=6
**Role:** Primary food engine. Pawns pay for themselves and feed the network.
**Strategy:** Greedy yield-maximizer. Place on the highest food-tier hex visible.
- Break-even: tier ≥ 2 (1.25×2=2.5 > diet 2). Tier 5+ generates real surplus.
- Lifespan ROI: 6 turns × (harvest - diet) - 7 cost. Tier 4 → 6×(5-2)-7 = +11 net.
- No connectivity concern — pawns are plentiful and cheap enough to lose.
- Drain 0.25 is negligible; it's a side effect, not a goal.
**Why greedy works:** Pawns are the squad's income. Every turn without a pawn
  harvesting is food left on the table. Fancy positioning loses to "biggest number."

### ♝ Bishop — "Swiss Army Knife"  
**Profile:** diet=3, cost=14, harvest=0.75, drain=1.5, sight=base+2, lifespan=4
**Role:** Adapts to what the network needs: relay, harvester, or frontier scout.
**Strategy:** Auto-detect role from network health, then score candidates.
- **Relay** (fragile network): weight allies_seen ×15. Bridge articulation points.
- **Harvester** (food tight): weight lifetime_net ×8. Need tier ≥ 4 to break even
  (0.75×4=3.0 = diet). Tier 5+ over 4 turns: 4×(3.75-3)-14 = -11... still negative.
  Bishops are marginal harvesters — this role is damage control, not profit.
- **Frontier** (healthy): weight new_land ×6. The +2 sight is the best in the roster
  for revealing new hexes, unlocking better pawn placements next turn.
**Why auto-detect:** 4-turn lifespan means you can't afford a bad role pick. A relay
  bishop in a healthy network wastes 14 food; a frontier bishop in a fragile network
  lets the chain snap. Reading the graph tells you what's actually needed.

### ♞ Knight — "Guided Missile"
**Profile:** diet=2, cost=6, harvest=0.0, drain=3.0, sight=base, lifespan=1
**Role:** One-shot drain strike against enemy clusters. Fire and forget.
**Strategy:** ROI-gated targeting. Only deploy if drain value > cost + diet.
- Minimum viable target: 3 enemies hit → 9 drain dealt for 8 food spent.
- Target weighting: kings ×20, queens ×10, bishops ×4, pawns ×2. Aim for HVTs.
- Rotation search: worth rotating an ally to unlock a better firing angle, since
  the knight dies anyway — the ally's facing can be fixed next turn.
- ROI gate prevents waste: if no cluster is reachable, returns None and saves food.
**Why ROI-gated:** Knights are pure consumables. Without the gate, the AI would
  fire knights at lone pawns (3 drain for 8 cost = net loss). The gate ensures
  every knight deployed is a profitable trade.

### ♛ Queen — "Fortress"
**Profile:** diet=5, cost=27, harvest=0.0, drain=4.25, sight=base, lifespan=8
**Role:** Area denial and king protection. Defensive anchor with offensive upside.
**Strategy:** Stance-based scoring (defend vs attack).
- **Defend** (default): allies_covered ×3, king_distance ×-4, king_visible +5 bonus.
  The queen's 4.25 drain makes approaching the king extremely costly for enemies.
  Placed 1-2 hexes from king, she creates a "no-go zone."
- **Attack**: enemies_caught × drain ×3. Used when ahead on food and wanting to
  close out a weakened opponent.
- Affordability gate: king needs pantry > cost + 10 (= 37). Queens are expensive;
  deploying one while food-starved can cascade into starvation deaths.
**Why fortress-first:** At cost=27 and 8-turn lifespan, the queen is the biggest
  investment. Losing the king ends the game; the queen's drain aura is the best
  insurance policy. Offensive value is a bonus, not the primary mission.

### ♚ King — "Treasury"  
**Profile:** diet=2, cost=0, harvest=1.0, drain=0.5, sight=base+1, lifespan=∞
**Role:** Network hub and food bank. All surplus flows to the king's pantry.
**No best_move needed:** Placed at game start via find_starts(). Facing is set by
  optimize_facings() to cover the most allies.
- The king's pantry IS the squad's bank account. Every placement cost comes from here.
- harvest=1.0 means the king is a decent earner on high-tier hexes (often tier 0
  at start positions though — find_starts prioritizes *regional* richness).
- +1 sight helps maintain connectivity as the network grows.


- **diet** — upkeep per turn while on board
- **cost** — one-time placement cost (deducted from king's pantry)
- **harvest** — multiplier × tile food tier = food earned per turn
- **drain** — cost imposed on EACH enemy piece caught in this piece's sight cone
- **bonus_sight** — extra sight range beyond elevation-based default

### LIFESPAN
Turns before a piece expires: PAWN=4, BISHOP=3, KNIGHT=3, QUEEN=2, KING=999.

### Squad
A player's army.
- `pieces: list[Piece]` — all pieces (placed and unplaced)
- `name, animal, flag` — identity
- No `storage` field — **king's pantry IS the squad treasury**

Key methods:
- `to_nx() → nx.DiGraph` — the central sight/flow graph (see below)
- `resolve_flow()` — apply max-flow food sharing via nx
- `best_pawn_move(parts) → Move` — greedy: find highest-yield hex in/near sight
- `king_alive() → bool`
- `tick_lifespan() → list[Piece]` — increment tenure, evict expired
- `supply_bottlenecks() → list[Piece]` — articulation points in sight graph
- `isolated_pieces() → list[Piece]` — pieces not connected to king's component
- `food_sinks() / food_sources()` — pieces that consume vs produce

---

## The Sight/Supply Graph (NetworkX)

`Squad.to_nx()` builds a **directed graph** where:
- **Nodes** = placed pieces, with attributes: `piece`, `piece_type`, `harvest` (pre-computed), `diet`, `drain`, `cost`, `pantry`, `hex_idx`, `is_sink`, `is_king`
- **Edges** = A → B means piece A can **see** piece B (B is in A's sight cone)
- Edges represent potential food transfer paths

### Key graph queries:
- `nx.articulation_points(G.to_undirected())` — supply bottlenecks
- `nx.has_path(G.to_undirected(), a, b)` — connectivity check
- `nx.maximum_flow(F, 'S', 'T')` — food distribution (super-source/super-sink formulation)
- `nx.node_connected_component(G, king_id)` — king's connected component

### Flow algorithm (`resolve_flow`):
1. Build auxiliary flow graph `F` with super-source `S` and super-sink `T`
2. Pieces with `pantry > 0` get edge `S → piece` (capacity = surplus)
3. Pieces with `pantry < 0` get edge `piece → T` (capacity = deficit)
4. Sight edges get capacity 999 (unlimited relay), bidirectional
5. Run `nx.maximum_flow(F, 'S', 'T')`
6. Apply flow: donors lose, receivers gain
7. Remaining surplus from connected pieces flows to king

---

## Moves

### Move
- `rotations: list[Rotation]` — (piece, new_facing) pairs for already-placed pieces
- `placement: Placement | None` — (piece, MapPlace) for a new piece entering the board

### Validation (`Move.from_squad`):
1. Rotated pieces must belong to squad and be placed
2. Placed piece must belong to squad and NOT be placed yet
3. A piece can't appear in both rotation and placement
4. First piece placed must be KING
5. Placement target must be in **post-rotation** sight of at least one ally
6. Target hex must not be occupied by any piece

### MoveLog
Records all moves as a DataFrame: turn, squad, piece, action (rotate/place), detail.

---

## Turn Processing

### TurnProcessor.resolve_move(squad, move) → dict

Phase order:
1. **Validate & apply** move (rotations + placement)
2. **Placement cost** deducted from king's pantry; new piece gets 75% of cost as starting pantry
3. **Tick lifespans** — evict expired pieces
4. **Harvest** — each piece earns `harvest_mult × tile_tier`
5. **Diet** — each piece pays its diet from own pantry
6. **Enemy drain** — each of our pieces drains enemy pieces in its cone
7. **Network flow** — `Squad.to_nx()` + `resolve_flow()` redistributes food via max-flow
8. **Starvation** — pieces still in deficit (except king) are removed, prioritized by: worst deficit → lowest rank → furthest from king
9. **Win check** — if king is dead, squad is eliminated; last king standing wins

### Turn order between squads:
- You move → economy resolves
- Enemy A moves → economy resolves
- Enemy B moves → economy resolves
- Back to you (you've taken N-1 economy hits between your moves)

---

## Food Economy Mental Model

```
   🌾 Pawns (harvest oasis tiles)
       ↓ sight edge (facing toward bishop)
   ♝ Bishops (relay + partial harvest)
       ↓ sight edge (facing toward king)
   ♚ King (treasury = pantry, feeds sinks)
       ↓ withdrawal
   ♞ Knights / ♛ Queens (sinks: drain enemies, no harvest)
```

- **Sources**: Pawns and Bishops on fertile tiles (harvest > diet)
- **Relays**: Any piece that sees both upstream and downstream
- **Sinks**: King (treasury), Queens, Knights (harvest = 0)
- **Cut a link**: downstream pieces lose access to shared food, must survive on own harvest
- **Drain**: each piece imposes `drain` cost on every enemy in its cone — stacks across multiple pieces

---

## Terrain & Food Yield

### FoodYield
Computes per-hex food tiers (0–7) from three factors:
- **Temperature** (climate zone → curve): subtropical peak, tundra low
- **Water** (precipitation + river flow, log-scale diminishing returns)
- **Soil** (type → fertility multiplier: alluvial best, granite worst)

Formula: `raw = temp_f × water_f × soil_f`, normalized to 0–1, quantized to tiers.

### find_starts
Finds balanced king placements:
1. Score each hex = sum of food tiers in ring of radius `ring_radius`
2. Take top `top_k` candidates
3. Try combinations, checking: minimum hex distance, land-connected, balance tolerance (spread relative to median)
4. Returns most balanced valid set

---

## Overlay System (SVG Rendering)

`TerrainDisplay` composites multiple `OverlaySpec` layers by priority:
- `TerrainOverlay` / `CreamOverlay` — base terrain colors
- `RiverOverlay` — water features
- `FoodOverlay` — dot-density pattern showing food yield tiers
- `FogOverlay` — whites out hexes outside a region (fog of war)
- `LanternOverlay` — sight cone visualization per piece
- `PieceOverlay` — chess piece SVG glyphs on the map
- `FoodNetworkOverlay` — sight graph edges, bottleneck markers, surplus/deficit indicators
- `OceanWaveOverlay` — decorative ocean waves

Each overlay gets a `GameContext` with: terrain, grid, builder, basins, pieces, squads.

---

## Pyomo Integration Notes (Future)

The `to_nx()` graph maps directly to Pyomo:
- `G.nodes` → `model.NODES` (Set)
- `G.edges` → `model.ARCS` (Set)
- Node attributes → `model.harvest`, `model.diet`, `model.drain` (Param dicts)
- Flow conservation: `inflow + harvest >= diet + outflow` per node
- Balance constraints: steady-state food across squads within tolerance
- The flow formulation is a standard min-cost network flow problem


In [ ]:
# After pawn rounds...
for squad in myStuff.squads:
    move = squad.best_queen_move(myStuff)
    if move:
        summary = tp.resolve_move(squad, move)
        print(f"  {squad.name}: 👑→♛ draining {summary.get('food_drain_dealt', 0)} food")


In [ ]:
for squad in myStuff.squads:
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed: continue
    print(f"\n{squad.name}:")
    for p in placed:
        idx = myStuff.hp2i(p.place.location)
        tier = int(myStuff.fy.tiers[idx])
        in_enemy_cones = 0
        for esq in myStuff.squads:
            if esq is squad: continue
            for ep in esq.pieces:
                if ep.place is not None and idx in ep.sight().hexes:
                    in_enemy_cones += 1
        drain_taken = sum(ep.food.drain for esq in myStuff.squads if esq is not squad
                         for ep in esq.pieces if ep.place is not None 
                         and idx in ep.sight().hexes)
        net = int(p.food.harvest * tier) - p.food.diet - int(drain_taken)
        print(f"  {p.type.icon} {p.name}: pantry={p.place.pantry}, "
              f"tier={tier}, drain_taken={drain_taken:.0f}, net/turn={net:+d}"
              f"{' ⚠️' if p.place.pantry + net < 0 else ''}")


## Placement strategy

In [ ]:
@dataclass
class PlacementStrategy:
    """Squad personality that shapes piece priority and stance thresholds."""
    name: str
    # Stance thresholds (how easily this personality shifts)
    defend_threat: int = 2       # enemy pieces near king to trigger defend
    attack_pantry: int = 60      # min king pantry to go offensive
    attack_placed: int = 4       # min pieces on board before attacking
    # Knight eagerness: lower = fires more readily
    knight_roi_mult: float = 1.0  # multiplied into knight ROI gate
    
    def assess(self, squad: Squad, parts: GameParts) -> str:
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return "defend"
        
        pantry = king.place.pantry
        placed = [p for p in squad.pieces if p.place is not None]
        G = squad.to_nx()
        U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
        bottlenecks = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
        
        king_hp = king.place.location
        threat_count = sum(
            1 for sq in parts.squads if sq is not squad
            for p in sq.pieces if p.place is not None
            and parts.i2hp(parts.hp2i(p.place.location)).distance(king_hp) <= 4
        )
        
        if threat_count >= self.defend_threat or bottlenecks >= 2:
            return "defend"
        elif pantry > self.attack_pantry and len(placed) >= self.attack_placed:
            return "attack"
        else:
            return "expand"
    
    def next_move(self, squad: Squad, parts: GameParts) -> Move | None:
        stance = self.assess(squad, parts)
        order = self.PLAYBOOKS[self.name].get(stance, self.PLAYBOOKS["balanced"][stance])
        
        for method_name, kwargs in order:
            fn = getattr(squad, method_name)
            move = fn(parts, **kwargs) if kwargs else fn(parts)
            if move is not None:
                return move
        return None
    
    # --- Playbooks: (method_name, kwargs) per stance per personality ---
    PLAYBOOKS = {
        "expansive": {
            "expand":  [("best_pawn_move", {}), ("best_pawn_move", {}), 
                        ("best_bishop_move", {}), ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_bishop_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "attack":  [("best_pawn_move", {}), ("best_bishop_move", {}),
                        ("best_knight_move", {}), ("best_queen_move", {"stance": "attack"})],
        },
        "aggressive": {
            "expand":  [("best_queen_move", {"stance": "defend"}), ("best_knight_move", {}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "defend":  [("best_queen_move", {"stance": "defend"}), ("best_knight_move", {}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "attack":  [("best_knight_move", {}), ("best_queen_move", {"stance": "attack"}),
                        ("best_knight_move", {}), ("best_bishop_move", {})],
        },
        "balanced": {
            "expand":  [("best_pawn_move", {}), ("best_bishop_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_queen_move", {"stance": "defend"}), ("best_bishop_move", {}),
                        ("best_pawn_move", {})],
            "attack":  [("best_knight_move", {}), ("best_queen_move", {"stance": "attack"}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
        },
        "turtle": {
            "expand":  [("best_pawn_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_queen_move", {"stance": "defend"}), 
                        ("best_queen_move", {"stance": "defend"}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "attack":  [("best_bishop_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],  # turtles never really attack
        },
    }

# --- Factory presets ---
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6)
PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3, knight_roi_mult=0.7)
PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4)
PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99)


In [ ]:
@patch
def next_move(self: PlacementStrategy, squad: Squad, parts: GameParts) -> Move | None:
    target = getattr(squad, '_target', None)
    
    # Tick first, then check expiry
    if target is not None:
        target.tick()
    
    if target is None or target.is_expired:
        squad._target = squad.choose_target(parts, self)
        squad.optimize_facings(parts, max_iters=3)
    else:
        squad.orient_facings(parts)
    
    # Pick best move per playbook
    stance = self.assess(squad, parts)
    order = self.PLAYBOOKS[self.name].get(stance, self.PLAYBOOKS["balanced"][stance])
    for method_name, kwargs in order:
        move = getattr(squad, method_name)(parts, **kwargs) if kwargs else getattr(squad, method_name)(parts)
        if move is not None:
            return move
    return None


### Target ideas

I think these squads should have targets that they are going after. either
1. A king they want to go after
2. an oasis they want to conquer

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great idea — and you've already got `target: int = -1` on `Squad`, so the slot is ready. Let me sketch out a clean design for this.

Two target types make sense:

1. **KING** — project drain toward an enemy king (offensive)
2. **OASIS** — expand toward a high-yield hex cluster (economic)

Here's a proposed `SquadTarget` dataclass and a selection method:

```python
from enum import Enum

class TargetType(Enum):
    KING = "king"
    OASIS = "oasis"

@dataclass
class SquadTarget:
    type: TargetType
    hex_idx: int              # the focal hex (king's location or oasis center)
    value: float = 0.0        # score explaining why this was chosen
    enemy_squad: str = ""     # only for KING targets
    
    @property
    def label(self) -> str:
        if self.type == TargetType.KING:
            return f"⚔️ {self.enemy_squad} king @ {self.hex_idx}"
        return f"🌾 oasis @ {self.hex_idx} (value={self.value:.0f})"
```

Then a method on Squad to evaluate candidates:

```python
@patch
def choose_target(self: Squad, parts: GameParts, 
                  oasis_radius: int = 3) -> SquadTarget | None:
    """Pick the best target: weakest nearby king OR richest unclaimed oasis."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None
    
    my_hp = king.place.location
    candidates = []
    
    # --- Candidate kings (weighted by weakness × proximity) ---
    for sq in parts.squads:
        if sq is self: continue
        eking = next((p for p in sq.pieces 
                      if p.type == PieceType.KING and p.place is not None), None)
        if not eking: continue
        eidx = parts.hp2i(eking.place.location)
        dist = my_hp.distance(eking.place.location)
        if dist == 0: continue
        # Lower pantry + fewer placed pieces = weaker = juicier target
        placed = sum(1 for p in sq.pieces if p.place is not None)
        weakness = 1.0 / max(1, eking.place.pantry) + 1.0 / max(1, placed)
        score = weakness / dist  # closer + weaker = higher score
        candidates.append(SquadTarget(
            TargetType.KING, eidx, value=score, enemy_squad=sq.name))
    
    # --- Candidate oases (richest unclaimed hex clusters) ---
    tiers = parts.fy.tiers
    terr = parts.terr
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Score each land hex by its neighborhood richness
    for i in range(len(terr.elevations)):
        if terr.elevations[i] <= 0: continue
        ring = parts.grid.indices_in_range(i, oasis_radius)
        richness = sum(tiers[j] for j in ring 
                       if terr.elevations[j] > 0 and j not in occupied)
        if richness < 15: continue  # skip mediocre spots
        dist = my_hp.distance(parts.i2hp(i))
        if dist == 0: continue
        score = richness / dist
        candidates.append(SquadTarget(TargetType.OASIS, i, value=score))
    
    if not candidates: return None
    return max(candidates, key=lambda t: t.value)
```

A couple of design questions before I build this out:

- **How often should targets update?** Every turn, or sticky until achieved/invalidated (king dies, oasis captured)? Sticky feels more natural — squads commit to a plan.
- **How should the target influence moves?** I'm thinking a distance bias added to the existing scorer functions — hexes closer to the target get a bonus. That way the existing `best_*_move` methods don't need rewriting, just a nudge toward the target direction.

What's your preference on those two points?

So some points
1. choose target should very much depend on strategy - aggressive goes after kings, turtle buils nearby.
2. we might want a consolidate target (which could be no target at all) - bascially build up your local strength
3. changing target should be interruption driven. If you lose a piece someone gets their first. But it might be easier to build out if we go russian style and have 5 turn strategies
4. I think this should influnce rotations - you go where you aim.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love these points. Let me lay out the design:

**Three target types:**
- `KING` — offensive, project toward enemy king
- `OASIS` — expansionist, grow toward rich unclaimed territory
- `CONSOLIDATE` — no focal hex, just optimize locally (face allies, harvest nearby)

**5-turn strategy cycles** (Russian-style planning):
- Every 5 turns, squad picks a new target based on personality + situation
- Target is sticky within the cycle — pieces placed and rotated toward it
- Early re-evaluation if something big happens (king threatened, piece lost)

**Strategy × target mapping:**

| Strategy | Healthy | Threatened | Ahead |
|---|---|---|---|
| Aggressive | KING (weakest) | KING (nearest) | KING (weakest) |
| Expansive | OASIS (richest) | CONSOLIDATE | OASIS (nearest) |
| Balanced | OASIS (richest) | CONSOLIDATE | KING (weakest) |
| Turtle | CONSOLIDATE | CONSOLIDATE | OASIS (nearest) |

**Rotation influence** — this is the key behavioral change. When a squad has a target hex:
- `orient_facings` biases toward the target direction (not just source→sink)
- Pieces on the "frontier" (closest to target) face outward toward it
- Pieces behind the frontier face toward the frontier (relay chain)

Here's the code:

```python
@dataclass
class SquadTarget:
    type: TargetType
    hex_idx: int = -1         # -1 for CONSOLIDATE
    value: float = 0.0
    enemy_squad: str = ""
    turns_remaining: int = 5  # countdown within strategy cycle
    
    @property
    def label(self) -> str:
        if self.type == TargetType.CONSOLIDATE:
            return f"🏰 consolidate ({self.turns_remaining}t left)"
        if self.type == TargetType.KING:
            return f"⚔️ {self.enemy_squad} king @ {self.hex_idx} ({self.turns_remaining}t)"
        return f"🌾 oasis @ {self.hex_idx} ({self.turns_remaining}t)"
    
    @property
    def is_expired(self) -> bool:
        return self.turns_remaining <= 0
    
    def tick(self):
        self.turns_remaining = max(0, self.turns_remaining - 1)
```

```python
class TargetType(Enum):
    KING = "king"
    OASIS = "oasis"
    CONSOLIDATE = "consolidate"
```

Then the strategy-aware selection:

```python
@patch
def choose_target(self: Squad, parts: GameParts, 
                  strategy: PlacementStrategy,
                  oasis_radius: int = 3) -> SquadTarget:
    """Pick target based on strategy personality + board state."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return SquadTarget(TargetType.CONSOLIDATE)
    
    stance = strategy.assess(self, parts)
    my_hp = king.place.location
    
    # --- Strategy × stance → preferred target type ---
    PREFERENCE = {
        "aggressive": {"expand": "king",  "defend": "king",        "attack": "king"},
        "expansive":  {"expand": "oasis", "defend": "consolidate", "attack": "oasis"},
        "balanced":   {"expand": "oasis", "defend": "consolidate", "attack": "king"},
        "turtle":     {"expand": "consolidate", "defend": "consolidate", "attack": "oasis"},
    }
    preferred = PREFERENCE.get(strategy.name, PREFERENCE["balanced"])[stance]
    
    # --- Score candidate kings ---
    king_candidates = []
    for sq in parts.squads:
        if sq is self: continue
        eking = next((p for p in sq.pieces 
                      if p.type == PieceType.KING and p.place is not None), None)
        if not eking: continue
        eidx = parts.hp2i(eking.place.location)
        dist = my_hp.distance(eking.place.location)
        if dist == 0: continue
        placed = sum(1 for p in sq.pieces if p.place is not None)
        weakness = 1.0 / max(1, eking.place.pantry) + 1.0 / max(1, placed)
        score = weakness / dist
        king_candidates.append(SquadTarget(
            TargetType.KING, eidx, value=score, enemy_squad=sq.name))
    
    # --- Score candidate oases ---
    oasis_candidates = []
    tiers = parts.fy.tiers
    terr = parts.terr
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    for i in range(len(terr.elevations)):
        if terr.elevations[i] <= 0: continue
        ring = parts.grid.indices_in_range(i, oasis_radius)
        richness = sum(tiers[j] for j in ring 
                       if terr.elevations[j] > 0 and j not in occupied)
        if richness < 15: continue
        dist = my_hp.distance(parts.i2hp(i))
        if dist == 0: continue
        score = richness / dist
        oasis_candidates.append(SquadTarget(TargetType.OASIS, i, value=score))
    
    # --- Pick based on preference, with fallback ---
    if preferred == "king" and king_candidates:
        return max(king_candidates, key=lambda t: t.value)
    if preferred == "oasis" and oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    if preferred == "consolidate":
        return SquadTarget(TargetType.CONSOLIDATE)
    
    # Fallback chain: oasis → consolidate
    if oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    return SquadTarget(TargetType.CONSOLIDATE)
```

And the rotation influence — a revised `orient_facings` that respects the target:

```python
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Target-aware facing: frontier pieces face target, rear pieces relay."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    
    for piece in placed:
        if piece.place is None: continue
        my_hp = piece.place.location
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        # Score each direction
        best_dir, best_score = piece.place.facing, -999
        
        # Build scoring targets
        allies = [p for p in placed if p is not piece]
        ally_idxs = {parts.hp2i(p.place.location) for p in allies}
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in my_hp.field_of_view(d, eff))
            
            # Base: see allies (connectivity)
            score = len(ally_idxs & cone) * 3.0
            
            # Target bias: if we have a target, reward facing toward it
            if target_hp is not None:
                target_idx = parts.hp2i(target_hp)
                # Direct: target hex in cone
                if target_idx in cone:
                    score += 8.0
                # Indirect: cone extends toward target (angular alignment)
                cone_center = _cone_centroid(my_hp, d, eff, parts)
                if cone_center:
                    alignment = 1.0 / max(1, cone_center.distance(target_hp))
                    score += alignment * 5.0
            
            # Consolidate: just maximize ally visibility
            if target is None or target.type == TargetType.CONSOLIDATE:
                king_idx = parts.hp2i(king.place.location) if king else -1
                if king_idx in cone:
                    score += 5.0
            
            if score > best_score:
                best_score = score
                best_dir = d
        
        piece.place.facing = best_dir


def _cone_centroid(origin: HexPosition, facing: HexPosition, 
                   sight: int, parts: GameParts) -> HexPosition | None:
    """Average position of hexes in a field of view — the 'aim point'."""
    fov = origin.field_of_view(facing, sight)
    if not fov: return None
    qs = [h.q for h in fov]
    rs = [h.r for h in fov]
    ss = [h.s for h in fov]
    n = len(fov)
    return HexPosition(round(sum(qs)/n), round(sum(rs)/n), round(sum(ss)/n))
```

The wiring in `PlacementStrategy` would look like:

```python
@patch
def next_move(self: PlacementStrategy, squad: Squad, parts: GameParts) -> Move | None:
    # Refresh target every 5 turns or if expired
    target = getattr(squad, '_target', None)
    if target is None or target.is_expired:
        squad._target = squad.choose_target(parts, self)
    
    # Orient facings toward current target
    squad.orient_facings(parts)
    
    # Tick the target countdown
    if squad._target:
        squad._target.tick()
    
    # Then pick best move per playbook (existing logic)
    stance = self.assess(squad, parts)
    order = self.PLAYBOOKS[self.name].get(stance, self.PLAYBOOKS["balanced"][stance])
    for method_name, kwargs in order:
        move = getattr(squad, method_name)(parts, **kwargs) if kwargs else getattr(squad, method_name)(parts)
        if move is not None:
            return move
    return None
```

The key changes from what you had:

1. `_target` lives on the Squad instance (set by `choose_target`, read by `orient_facings` and scorers)
2. CONSOLIDATE = no target hex, just face allies — turtles love this
3. 5-turn cycle with `turns_remaining` countdown, re-evaluated when it hits 0
4. `orient_facings` now has a target bias term — pieces "lean" toward where the squad is heading

Want to wire this up and test it with the existing simulation loop?

Did I wire this in correctly

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Almost! The wiring between `choose_target` → `next_move` looks correct:

- `TargetType` enum ✅
- `SquadTarget` dataclass ✅  
- `choose_target(self, parts, strategy)` — correctly strategy-aware ✅
- `next_move` patch — calls `choose_target`, orients, ticks, then falls through to playbook ✅

**The one missing piece:** `orient_facings` is still the old version — it does source→sink / sink→source / king→centroid. It never reads `squad._target`. So you're choosing targets and ticking them down, but nobody's actually *aiming* at them.

That's the method that needs the target bias. Right now it's:

```python
# Current: purely role-based
for p in sources:
    p.place.facing = best_facing_toward(p, sink_targets)
```

It should be something like:

```python
# Target-aware: bias toward squad._target
target = getattr(self, '_target', None)
target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None

# Frontier pieces (closest to target) → face target
# Rear pieces → face toward frontier allies (relay chain)
# CONSOLIDATE → fall back to the old source→sink logic
```

Want me to write the updated `orient_facings` that reads `_target`?

There was an optimized version of orient_facing that was similar to a wherehouse location. It is expensive, but calling it when we do a plan might make sense. can you combine that into opimize facings. The objective function would change based upon strategy.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great idea — run the expensive optimization once when the plan is set (every 5 turns), then cheap `orient_facings` for incremental adjustments between plans.

The key change: `_score_network` becomes target-aware. Three objective functions:

- **KING target** — reward drain coverage on enemy pieces near target, penalize broken connectivity
- **OASIS target** — reward new land revealed toward target, reward harvest potential
- **CONSOLIDATE** — reward connectivity, ally coverage, minimize bottlenecks (the old logic)

```python
def _score_network(squad: Squad, parts: GameParts) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    G = squad.to_nx()
    if len(G) < 2: return 0.0
    
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return 0.0
    U = G.to_undirected()
    
    # --- Base: connectivity (always matters) ---
    king_comp = nx.node_connected_component(U, king_id)
    n_connected = len(king_comp)
    n_total = len(G)
    connectivity = (n_connected / n_total) * 100
    bottleneck_penalty = len(list(nx.articulation_points(U))) * 5
    edge_bonus = U.number_of_edges() * 0.5
    
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    # --- CONSOLIDATE or no target: old logic ---
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    
    if target.type == TargetType.KING:
        # --- KING target: maximize drain on enemies near target ---
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    ep_idx = parts.hp2i(ep.place.location)
                    if ep_idx in cone:
                        # Weight by proximity to target king
                        ep_hp = ep.place.location
                        dist_to_target = max(1, ep_hp.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        # Reward sight cones that reach toward the target
        projection = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            cone = p.sight().hexes
            if target.hex_idx in cone:
                projection += 10.0
            else:
                # Partial credit: closest cone hex to target
                if cone:
                    min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                    projection += 2.0 / max(1, min_dist)
        
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        # --- OASIS target: maximize new land revealed toward target ---
        land_set = {i for i in range(len(parts.terr.elevations)) 
                    if parts.terr.elevations[i] > 0}
        
        # Collective sight
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        
        new_land = len(all_visible & land_set)
        
        # Reward sight that extends toward the oasis
        projection = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            cone = p.sight().hexes
            if target.hex_idx in cone:
                projection += 10.0
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        projection += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        
        return base + new_land * 1.5 + projection * 2.0 + food_reach


@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Coordinate descent over facings using target-aware objective.
    
    Expensive — call when setting a new 5-turn plan, not every turn.
    Falls back to orient_facings for priority ordering of first pass.
    """
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2:
        return 0
    
    # Seed with cheap heuristic first
    self.orient_facings(parts)
    
    # Priority: optimize bottlenecks and frontier pieces first
    G = self.to_nx()
    U = G.to_undirected()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    bottlenecks = set(nx.articulation_points(U)) if len(G) > 1 else set()
    king_neighbors = set(U.neighbors(king_id)) if king_id else set()
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    def priority(p):
        nid = p.id
        is_bn = nid in bottlenecks
        is_kn = nid in king_neighbors
        # Frontier = closest to target (optimize these first for KING/OASIS)
        if target_hp:
            dist = p.place.location.distance(target_hp)
        else:
            dist = 999
        return (not is_bn, not is_kn, dist)
    
    total_changes = 0
    
    for _ in range(max_iters):
        changed = False
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            if piece.place is None: continue
            original = piece.place.facing
            best_facing = original
            best_score = _score_network(self, parts)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                s = _score_network(self, parts)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes
```

Then update `next_move` to use the expensive version only on plan changes:

```python
@patch
def next_move(self: PlacementStrategy, squad: Squad, parts: GameParts) -> Move | None:
    target = getattr(squad, '_target', None)
    
    if target is None or target.is_expired:
        # New plan — pick target and run expensive optimization
        squad._target = squad.choose_target(parts, self)
        squad.optimize_facings(parts, max_iters=3)
    else:
        # Mid-plan — cheap orientation only
        squad.orient_facings(parts)
        squad._target.tick()
    
    # Pick best move per playbook
    stance = self.assess(squad, parts)
    order = self.PLAYBOOKS[self.name].get(stance, self.PLAYBOOKS["balanced"][stance])
    for method_name, kwargs in order:
        move = getattr(squad, method_name)(parts, **kwargs) if kwargs else getattr(squad, method_name)(parts)
        if move is not None:
            return move
    return None
```

The pattern is:
- **Plan set** (every 5 turns): `choose_target` → `optimize_facings` (expensive, ~6 dirs × N pieces × 3 iters)  
- **Mid-plan** (turns 2–5): `orient_facings` (cheap heuristic) + `tick()`
- **Objective function** automatically shifts based on `_target.type` — aggressive squads optimize for drain projection, turtles optimize for connectivity

did I wire this in correctly

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It's 90% right — the wiring between `next_move` → `choose_target` → `optimize_facings` is solid, and the tick/expiry logic is correct (5 actual turns per plan).

**One gap:** `orient_facings` (cell `_f0e29adb`) is still the old role-based version — source→sink, sink→source, king→centroid. It never reads `self._target`. So on turns 2–5 of a plan, when `next_move` calls `orient_facings` instead of the expensive `optimize_facings`, newly placed pieces get oriented with zero awareness of the target.

That means:
- **Turn 1** (new plan): `optimize_facings` ✅ uses `_score_network` which reads `_target`
- **Turns 2–5**: `orient_facings` ❌ ignores `_target` entirely

The fix is to make `orient_facings` target-aware. Replace the `best_facing_toward` logic with something that also considers the target:

```python
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Cheap target-aware facing: blend ally connectivity with target projection."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    allies = {parts.hp2i(p.place.location) for p in placed}
    
    for piece in placed:
        my_hp = piece.place.location
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        best_dir, best_score = piece.place.facing, -999
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in my_hp.field_of_view(d, eff))
            
            # Connectivity: always valuable
            score = len((allies - {parts.hp2i(my_hp)}) & cone) * 3.0
            
            # King visibility bonus
            if king and king is not piece:
                if parts.hp2i(king.place.location) in cone:
                    score += 4.0
            
            # Target projection (the new part)
            if target_hp is not None:
                target_idx = parts.hp2i(target_hp)
                if target_idx in cone:
                    score += 8.0
                else:
                    # Partial: how far does cone extend toward target?
                    if cone:
                        min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                        score += 3.0 / max(1, min_dist)
            
            if score > best_score:
                best_score = score
                best_dir = d
        
        piece.place.facing = best_dir
```

This keeps it cheap (one pass, no re-scoring the whole network) but adds the target bias. The weight split is roughly:
- **CONSOLIDATE** (target_hp=None): pure connectivity + king visibility — same as before
- **KING/OASIS** (target_hp set): connectivity + lean toward target

Everything else in the wiring looks correct.

Is this better?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, this is wired correctly now. The three pieces fit together:

1. **`orient_facings`** (cell `_f0e29adb`) — now reads `self._target` and adds the target projection bias ✅
2. **`optimize_facings`** (cell `_81244b8f`) — seeds with `orient_facings`, then coordinate-descents using `_score_network` which is also target-aware ✅
3. **`next_move`** (cell `_8e511a835`) — tick first, re-plan if expired, cheap orient otherwise ✅

The lifecycle is clean:

| Call # | `turns_remaining` | What happens |
|---|---|---|
| 1 | None → 5 | `choose_target` + `optimize_facings` (expensive) |
| 2 | 5 → 4 | `orient_facings` (cheap) |
| 3 | 4 → 3 | `orient_facings` (cheap) |
| 4 | 3 → 2 | `orient_facings` (cheap) |
| 5 | 2 → 1 | `orient_facings` (cheap) |
| 6 | 1 → 0 | `choose_target` + `optimize_facings` (expensive, new plan) |

One small note: `_score_network` is a module-level function, not a method — make sure it's defined *before* `optimize_facings` in execution order. Looking at your cells, `_1253e1de` (where `_score_network` lives) runs before `_81244b8f` (where `optimize_facings` calls it), so that's fine.

Ready to test-run a full game with strategies + targets to see how the squads behave?

In [ ]:
def run_game(parts, strategies: dict[str, PlacementStrategy], rounds=15):
    """strategies maps squad.name → PlacementStrategy"""
    tp = TurnProcessor(parts, parts.squads)
    
    for r in range(rounds):
        if tp.game_over: break
        for squad in parts.squads:
            if not squad.king_alive(): continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                summary = tp.resolve_move(squad, move)
        
        # Optimize facings every 3 rounds
        if (r + 1) % 3 == 0:
            for squad in parts.squads:
                squad.optimize_facings(parts)
        
        if r % 5 == 0:
            print(f"\n--- Round {r} ---")
            print(tp.status())
    
    print(f"\n=== FINAL (round {tp.turn}) ===")
    print(tp.status())
    return tp


In [ ]:
# Fresh game
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

# Assign different personalities
personality_rotation = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
    PlacementStrategy.AGGRESSIVE,
]
strategies = {}
for i, sq in enumerate(myStuff.squads):
    strategies[sq.name] = personality_rotation[i % len(personality_rotation)]
    print(f"{sq.name}: {strategies[sq.name].name}")

tp = run_game(myStuff, strategies, rounds=15)

In [ ]:
import optuna

Lets think through how we would tune. We have the parameters of 
lifespan, diet, cost, harvest, drain, bonus sight.
There should be a minimal number of pawns we need to support a queen via a bishop. queens and knights should have good combat strength against bishops. Once a terriority has been "claimed" by a queen it should be hard to move in. a group of knights shouldn't be able to take out a queen unless they are flanking her.

Any other ideas we should put it?

In [ ]:
def score_lone_king_dies(parts):
    """A king alone on barren land with pantry 20 should die ~turn 10."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] == 0)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0
    
    death_turn = 20 / abs(net_per_turn)  # smaller starting pantry
    
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


This was our first model

```python
def apply_params(trial_or_dict):
    """Apply tuned params to global FOOD_TABLE and LIFESPAN. 
    Accepts an Optuna trial or a plain dict."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            # Pawn
            'pawn_diet':    t.suggest_int('pawn_diet', 1, 2),
            'pawn_cost':    t.suggest_int('pawn_cost', 3, 8),
            'pawn_harvest': t.suggest_float('pawn_harvest', 0.5, 1.5, step=0.25),
            'pawn_drain':   t.suggest_float('pawn_drain', 0.0, 1.0, step=0.25),
            'pawn_sight':   0,
            'pawn_life':    t.suggest_int('pawn_life', 3, 6),
            # Bishop
            'bishop_diet':    t.suggest_int('bishop_diet', 1, 3),
            'bishop_cost':    t.suggest_int('bishop_cost', 8, 16),
            'bishop_harvest': t.suggest_float('bishop_harvest', 0.25, 1.0, step=0.25),
            'bishop_drain':   t.suggest_float('bishop_drain', 0.5, 2.0, step=0.25),
            'bishop_sight':   t.suggest_int('bishop_sight', 1, 3),
            'bishop_life':    t.suggest_int('bishop_life', 3, 5),
            # Knight (raider: cheap, short-lived)
            'knight_diet':    t.suggest_int('knight_diet', 1, 3),
            'knight_cost':    t.suggest_int('knight_cost', 5, 12),
            'knight_harvest': 0.0,
            'knight_drain':   t.suggest_float('knight_drain', 1.0, 3.0, step=0.25),
            'knight_sight':   0,
            'knight_life':    t.suggest_int('knight_life', 1, 3),
            # Queen (fortress: expensive, long-lived)
            'queen_diet':    t.suggest_int('queen_diet', 3, 6),
            'queen_cost':    t.suggest_int('queen_cost', 15, 30),
            'queen_harvest': 0.0,
            'queen_drain':   t.suggest_float('queen_drain', 2.0, 5.0, step=0.25),
            'queen_sight':   t.suggest_int('queen_sight', 0, 2),
            'queen_life':    t.suggest_int('queen_life', 4, 8),
            # King (fixed)
            'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
            'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
        }
    
    global FOOD_TABLE, LIFESPAN
    for pt, key in [(PieceType.PAWN, 'pawn'), (PieceType.BISHOP, 'bishop'),
                    (PieceType.KNIGHT, 'knight'), (PieceType.QUEEN, 'queen'),
                    (PieceType.KING, 'king')]:
        FOOD_TABLE[pt] = FoodStats(
            diet=p[f'{key}_diet'], cost=p[f'{key}_cost'],
            harvest=p[f'{key}_harvest'], drain=p[f'{key}_drain'],
            bonus_sight=p[f'{key}_sight'])
        LIFESPAN[pt] = p[f'{key}_life']
    
    return p


def score_3pawns_support_queen(parts):
    """Objective 1: 3 pawns on tier 6 tiles + 1 bishop + 1 queen, 
    fed through the network. Queen should survive 'queen_life' turns."""
    # Find good tiles near a cluster
    tiers = parts.fy.tiers
    terr = parts.terr
    grid = parts.grid
    
    # Find a tier 6+ hex for king
    king_idx = next(i for i in range(len(tiers)) 
                    if terr.elevations[i] > 0 and tiers[i] >= 4)
    king_hp = parts.i2hp(king_idx)
    
    # Find nearby good hexes
    nearby = grid.indices_in_range(king_idx, 4)
    good = [i for i in nearby 
            if terr.elevations[i] > 0 and tiers[i] >= 5 and i != king_idx]
    good.sort(key=lambda i: -tiers[i])
    
    if len(good) < 4:
        return 0.0  # can't set up scenario
    
    # Make a temporary squad
    flag = parts.flags[0]
    pieces = [
        Piece(id='test_king', flag=flag, type=PieceType.KING, name='K'),
        Piece(id='test_p1', flag=flag, type=PieceType.PAWN, name='P1'),
        Piece(id='test_p2', flag=flag, type=PieceType.PAWN, name='P2'),
        Piece(id='test_p3', flag=flag, type=PieceType.PAWN, name='P3'),
        Piece(id='test_b1', flag=flag, type=PieceType.BISHOP, name='B1'),
        Piece(id='test_q1', flag=flag, type=PieceType.QUEEN, name='Q1'),
    ]
    sq = Squad(pieces=pieces, name='test_3p', flag=flag)
    
    # Place them
    pieces[0].place = parts.location_by_index(king_idx)
    pieces[0].place.pantry = 50
    for i in range(3):
        pieces[i+1].place = parts.location_by_index(good[i])
        pieces[i+1].place.pantry = 5
    pieces[4].place = parts.location_by_index(good[3])
    pieces[4].place.pantry = 5
    # Queen adjacent to bishop
    q_candidates = [j for j in grid.indices_in_range(good[3], 2)
                    if terr.elevations[j] > 0 
                    and j not in {king_idx} | set(good[:4])]
    if not q_candidates:
        return 0.0
    pieces[5].place = parts.location_by_index(q_candidates[0])
    pieces[5].place.pantry = 10
    
    # Point everyone toward each other
    old_squads = parts.squads
    parts.squads = [sq]
    sq.orient_facings(parts)
    
    # Simulate turns — count how many turns queen survives
    queen_survived = 0
    for turn in range(LIFESPAN[PieceType.QUEEN] + 2):
        placed = [p for p in pieces if p.place is not None]
        for p in placed:
            p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
        if len(placed) > 1:
            sq.resolve_flow()
        expired = sq.tick_lifespan()
        
        # Check queen
        if pieces[5].place is not None and pieces[5].place.pantry >= 0:
            queen_survived += 1
        else:
            break
    
    # Clean up
    for p in pieces:
        p.place = None
    parts.squads = old_squads
    
    # Score: queen should survive her full lifespan
    target = LIFESPAN[PieceType.QUEEN]
    return min(queen_survived / target, 1.0)


def score_2knights_beat_queen(parts):
    """Objective 2: 2 knights flanking from different directions 
    should out-drain a queen (combined drain > queen drain)."""
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    
    # 2 knights flanking = 2 × knight_drain vs queen's 1 × queen_drain
    # Knights face different directions so queen can only drain 1 back
    knight_total_drain = 2 * kd       # both drain the queen
    queen_drain_back = qd * 1          # queen only sees 1 knight (facing one way)
    
    net_advantage = knight_total_drain - queen_drain_back
    # Want net_advantage > 0, ideally > 1
    if net_advantage <= 0:
        return 0.0
    return min(net_advantage / 2.0, 1.0)


def score_queen_beats_knight(parts):
    """Objective 3: 1v1, queen out-drains knight."""
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    
    # Head-to-head, both see each other
    # Queen wins if her drain > knight's drain
    advantage = qd - kd
    if advantage <= 0:
        return 0.0
    return min(advantage / 2.0, 1.0)


def score_bottomup_beats_topdown(parts):
    """Objective 4: Run a game, bottom-up strategy should outlast top-down."""
    # Fresh squads on the existing terrain
    parts.loadSquads(year=1950, seed=42, levels=2)
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = 100
    
    strategies = {
        parts.squads[0].name: PlacementStrategy.BOTTOM_UP,
        parts.squads[1].name: PlacementStrategy.TOP_DOWN,
    }
    
    tp = TurnProcessor(parts, parts.squads)
    bu_alive_turns = 0
    td_alive_turns = 0
    
    for r in range(20):
        if tp.game_over:
            break
        for squad in parts.squads:
            if not squad.king_alive():
                continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                try:
                    tp.resolve_move(squad, move)
                except MoveError:
                    pass
        
        if parts.squads[0].king_alive():
            bu_alive_turns += 1
        if parts.squads[1].king_alive():
            td_alive_turns += 1
    
    # Clean up pieces
    for sq in parts.squads:
        for p in sq.pieces:
            p.place = None
    
    # Score: bottom-up should survive longer
    if bu_alive_turns > td_alive_turns:
        return 1.0
    elif bu_alive_turns == td_alive_turns:
        return 0.5
    return 0.0


def score_lone_king_dies(parts):
    """Objective 5: A king with starting pantry 100, no pieces, 
    should die around turn 8-12."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    # King on a mediocre tile (tier 2)
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] <= 2)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0  # king never dies alone — bad
    
    death_turn = 100 / abs(net_per_turn)
    
    # Want death around turn 8-12
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


def objective(trial):
    """Combined Optuna objective — maximize sum of all sub-scores."""
    p = apply_params(trial)
    
    # Use a consistent terrain
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    parts.loadSquads(year=1950, seed=23, levels=2)
    
    s1 = score_3pawns_support_queen(parts)
    s2 = score_2knights_beat_queen(parts)
    s3 = score_queen_beats_knight(parts)
    s4 = score_bottomup_beats_topdown(parts)
    s5 = score_lone_king_dies(parts)
    
    # Weighted sum
    total = (s1 * 3.0 +   # sustain is most important
             s2 * 2.0 +   # flanking dynamic
             s3 * 2.0 +   # queen dominance
             s4 * 2.0 +   # strategy balance  
             s5 * 1.0)    # king pressure
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('strat', s4)
    trial.set_user_attr('king_dies', s5)
    
    return total


# Create and run the study
study = optuna.create_study(direction='maximize', study_name='hex_balance')
study.optimize(objective, n_trials=80, show_progress_bar=True)

# Show results
print(f"\nBest score: {study.best_value:.2f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"\nSub-scores:")
for k in ['sustain', 'flank', 'queen_1v1', 'strat', 'king_dies']:
    print(f"  {k}: {study.best_trial.user_attrs[k]:.2f}")
```

This was us checking parameter
```python
# Apply best params and print the new tables
params = study.best_params | {
    'pawn_sight': 0, 'knight_harvest': 0.0, 'knight_sight': 0,
    'queen_harvest': 0.0,
    'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
    'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
}
apply_params(params)
print("FOOD_TABLE:")
for pt, fs in FOOD_TABLE.items():
    print(f"  {pt.value:8s}: diet={fs.diet} cost={fs.cost} harvest={fs.harvest} "
          f"drain={fs.drain} sight={fs.bonus_sight} life={LIFESPAN[pt]}")
```

FOOD_TABLE:
  pawn    : diet=2 cost=7 harvest=1.25 drain=0.25 sight=0 life=6
  bishop  : diet=3 cost=14 harvest=0.75 drain=1.5 sight=2 life=4
  queen   : diet=5 cost=27 harvest=0.0 drain=4.25 sight=0 life=8
  knight  : diet=2 cost=6 harvest=0.0 drain=3.0 sight=0 life=1
  king    : diet=2 cost=0 harvest=1.0 drain=0.5 sight=1 life=999

In [ ]:
def apply_params(trial_or_dict):
    """Apply tuned params to global FOOD_TABLE and LIFESPAN. 
    Accepts an Optuna trial or a plain dict."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            # Pawn
            'pawn_diet':    t.suggest_int('pawn_diet', 1, 2),
            'pawn_cost':    t.suggest_int('pawn_cost', 3, 8),
            'pawn_harvest': t.suggest_float('pawn_harvest', 0.5, 1.5, step=0.25),
            'pawn_drain':   t.suggest_float('pawn_drain', 0.0, 1.0, step=0.25),
            'pawn_sight':   0,
            'pawn_life':    t.suggest_int('pawn_life', 3, 6),
            # Bishop
            'bishop_diet':    t.suggest_int('bishop_diet', 1, 3),
            'bishop_cost':    t.suggest_int('bishop_cost', 8, 16),
            'bishop_harvest': t.suggest_float('bishop_harvest', 0.25, 1.0, step=0.25),
            'bishop_drain':   t.suggest_float('bishop_drain', 0.5, 2.0, step=0.25),
            'bishop_sight':   t.suggest_int('bishop_sight', 1, 3),
            'bishop_life':    t.suggest_int('bishop_life', 3, 5),
            # Knight (raider: cheap, short-lived)
            'knight_diet':    t.suggest_int('knight_diet', 1, 3),
            'knight_cost':    t.suggest_int('knight_cost', 5, 12),
            'knight_harvest': 0.0,
            'knight_drain':   t.suggest_float('knight_drain', 1.0, 3.0, step=0.25),
            'knight_sight':   0,
            'knight_life':    t.suggest_int('knight_life', 1, 3),
            # Queen (fortress: expensive, long-lived)
            'queen_diet':    t.suggest_int('queen_diet', 3, 6),
            'queen_cost':    t.suggest_int('queen_cost', 15, 30),
            'queen_harvest': 0.0,
            'queen_drain':   t.suggest_float('queen_drain', 2.0, 5.0, step=0.25),
            'queen_sight':   t.suggest_int('queen_sight', 0, 2),
            'queen_life':    t.suggest_int('queen_life', 4, 8),
            # King (fixed)
            'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
            'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
        }
    
    global FOOD_TABLE, LIFESPAN
    for pt, key in [(PieceType.PAWN, 'pawn'), (PieceType.BISHOP, 'bishop'),
                    (PieceType.KNIGHT, 'knight'), (PieceType.QUEEN, 'queen'),
                    (PieceType.KING, 'king')]:
        FOOD_TABLE[pt] = FoodStats(
            diet=p[f'{key}_diet'], cost=p[f'{key}_cost'],
            harvest=p[f'{key}_harvest'], drain=p[f'{key}_drain'],
            bonus_sight=p[f'{key}_sight'])
        LIFESPAN[pt] = p[f'{key}_life']
    
    return p


def score_3pawns_support_queen(parts):
    """Objective 1: 3 pawns on tier 6 tiles + 1 bishop + 1 queen, 
    fed through the network. Queen should survive 'queen_life' turns."""
    # Find good tiles near a cluster
    tiers = parts.fy.tiers
    terr = parts.terr
    grid = parts.grid
    
    # Find a tier 6+ hex for king
    king_idx = next(i for i in range(len(tiers)) 
                    if terr.elevations[i] > 0 and tiers[i] >= 4)
    king_hp = parts.i2hp(king_idx)
    
    # Find nearby good hexes
    nearby = grid.indices_in_range(king_idx, 4)
    good = [i for i in nearby 
            if terr.elevations[i] > 0 and tiers[i] >= 5 and i != king_idx]
    good.sort(key=lambda i: -tiers[i])
    
    if len(good) < 4:
        return 0.0  # can't set up scenario
    
    # Make a temporary squad
    flag = parts.flags[0]
    pieces = [
        Piece(id='test_king', flag=flag, type=PieceType.KING, name='K'),
        Piece(id='test_p1', flag=flag, type=PieceType.PAWN, name='P1'),
        Piece(id='test_p2', flag=flag, type=PieceType.PAWN, name='P2'),
        Piece(id='test_p3', flag=flag, type=PieceType.PAWN, name='P3'),
        Piece(id='test_b1', flag=flag, type=PieceType.BISHOP, name='B1'),
        Piece(id='test_q1', flag=flag, type=PieceType.QUEEN, name='Q1'),
    ]
    sq = Squad(pieces=pieces, name='test_3p', flag=flag)
    
    # Place them
    pieces[0].place = parts.location_by_index(king_idx)
    pieces[0].place.pantry = 50
    for i in range(3):
        pieces[i+1].place = parts.location_by_index(good[i])
        pieces[i+1].place.pantry = 5
    pieces[4].place = parts.location_by_index(good[3])
    pieces[4].place.pantry = 5
    # Queen adjacent to bishop
    q_candidates = [j for j in grid.indices_in_range(good[3], 2)
                    if terr.elevations[j] > 0 
                    and j not in {king_idx} | set(good[:4])]
    if not q_candidates:
        return 0.0
    pieces[5].place = parts.location_by_index(q_candidates[0])
    pieces[5].place.pantry = 10
    
    # Point everyone toward each other
    old_squads = parts.squads
    parts.squads = [sq]
    sq.orient_facings(parts)
    
    # Simulate turns — count how many turns queen survives
    queen_survived = 0
    for turn in range(LIFESPAN[PieceType.QUEEN] + 2):
        placed = [p for p in pieces if p.place is not None]
        for p in placed:
            p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
        if len(placed) > 1:
            sq.resolve_flow()
        expired = sq.tick_lifespan()
        
        # Check queen
        if pieces[5].place is not None and pieces[5].place.pantry >= 0:
            queen_survived += 1
        else:
            break
    
    # Clean up
    for p in pieces:
        p.place = None
    parts.squads = old_squads
    
    # Score: queen should survive her full lifespan
    target = LIFESPAN[PieceType.QUEEN]
    return min(queen_survived / target, 1.0)


def score_2knights_beat_queen(parts):
    """Objective 2: 2 knights flanking from different directions 
    should out-drain a queen (combined drain > queen drain)."""
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    
    # 2 knights flanking = 2 × knight_drain vs queen's 1 × queen_drain
    # Knights face different directions so queen can only drain 1 back
    knight_total_drain = 2 * kd       # both drain the queen
    queen_drain_back = qd * 1          # queen only sees 1 knight (facing one way)
    
    net_advantage = knight_total_drain - queen_drain_back
    # Want net_advantage > 0, ideally > 1
    if net_advantage <= 0:
        return 0.0
    return min(net_advantage / 2.0, 1.0)


def score_queen_beats_knight(parts):
    """Objective 3: 1v1, queen out-drains knight."""
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    
    # Head-to-head, both see each other
    # Queen wins if her drain > knight's drain
    advantage = qd - kd
    if advantage <= 0:
        return 0.0
    return min(advantage / 2.0, 1.0)


def score_bottomup_beats_topdown(parts):
    """Objective 4: Run a game, bottom-up strategy should outlast top-down."""
    # Fresh squads on the existing terrain
    parts.loadSquads(year=1950, seed=42, levels=2)
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = 100
    
    strategies = {
        parts.squads[0].name: PlacementStrategy.BOTTOM_UP,
        parts.squads[1].name: PlacementStrategy.TOP_DOWN,
    }
    
    tp = TurnProcessor(parts, parts.squads)
    bu_alive_turns = 0
    td_alive_turns = 0
    
    for r in range(20):
        if tp.game_over:
            break
        for squad in parts.squads:
            if not squad.king_alive():
                continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                try:
                    tp.resolve_move(squad, move)
                except MoveError:
                    pass
        
        if parts.squads[0].king_alive():
            bu_alive_turns += 1
        if parts.squads[1].king_alive():
            td_alive_turns += 1
    
    # Clean up pieces
    for sq in parts.squads:
        for p in sq.pieces:
            p.place = None
    
    # Score: bottom-up should survive longer
    if bu_alive_turns > td_alive_turns:
        return 1.0
    elif bu_alive_turns == td_alive_turns:
        return 0.5
    return 0.0


def score_lone_king_dies(parts):
    """Objective 5: A king with starting pantry 100, no pieces, 
    should die around turn 8-12."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    # King on a mediocre tile (tier 2)
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] <= 2)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0  # king never dies alone — bad
    
    death_turn = 100 / abs(net_per_turn)
    
    # Want death around turn 8-12
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)

In [ ]:
def score_lone_king_dies(parts):
    """A king alone on barren land with pantry 20 should die ~turn 10."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] == 0)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0
    
    death_turn = 20 / abs(net_per_turn)  # smaller starting pantry
    
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


In [ ]:
def make_parts(seed):
    """Create a fresh GameParts with a specific seed."""
    random.seed(seed)
    np.random.seed(seed)
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    return parts

def objective_multi(trial):
    p = apply_params(trial)
    
    scores = {k: [] for k in ['sustain', 'flank', 'queen_1v1', 'strat', 'king_dies']}
    
    for seed in MAP_SEEDS:
        parts = make_parts(seed)
        parts.loadSquads(year=1950, seed=seed, levels=2)
        
        scores['sustain'].append(score_3pawns_support_queen(parts))
        scores['flank'].append(score_2knights_beat_queen(parts))
        scores['queen_1v1'].append(score_queen_beats_knight(parts))
        scores['strat'].append(score_bottomup_beats_topdown(parts))
        scores['king_dies'].append(score_lone_king_dies(parts))
    
    s1 = sum(scores['sustain']) / len(MAP_SEEDS)
    s2 = sum(scores['flank']) / len(MAP_SEEDS)
    s3 = sum(scores['queen_1v1']) / len(MAP_SEEDS)
    s4 = sum(scores['strat']) / len(MAP_SEEDS)
    s5 = sum(scores['king_dies']) / len(MAP_SEEDS)
    
    total = (s1 * 3.0 + s2 * 2.0 + s3 * 2.0 + s4 * 2.0 + s5 * 1.0)
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('strat', s4)
    trial.set_user_attr('king_dies', s5)
    trial.set_user_attr('strat_std', float(np.std(scores['strat'])))
    trial.set_user_attr('sustain_std', float(np.std(scores['sustain'])))
    
    return total

simulator code

```python
# Pre-build map cache (run once)
MAP_CACHE = {}
for seed in MAP_SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    MAP_CACHE[seed] = parts
    print(f"  Cached seed {seed}")

def fresh_game(seed, n_squads=2):
    """Return a GameParts with cached terrain + fresh squads."""
    cached = MAP_CACHE[seed]
    # Reattach fresh squads (terrain/grid/fy are untouched)
    cached.loadSquads(year=1950, seed=seed, levels=n_squads)
    
    starts = cached.find_starts(n=n_squads, ring_radius=3, min_distance=6)
    for squad, start_idx in zip(cached.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = cached.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(cached)
        king.place.pantry = 100
    
    return cached
```


This was the long trial

```python
MAP_SEEDS = [77, 101, 256]
# Expected matchup matrix: row beats column by this margin
# Positive = row favored, negative = column favored
# Values are target win rates for the ROW player
MATCHUP_TARGETS = {
    ('expansive',  'aggressive'): 0.35,  # aggressive should win ~65%
    ('expansive',  'balanced'):   0.50,
    ('expansive',  'turtle'):     0.70,  # expansive should win ~70%
    ('aggressive', 'balanced'):   0.50,
    ('aggressive', 'turtle'):     0.30,  # turtle should win ~70%
    ('balanced',   'turtle'):     0.50,
}

def score_rps_balance(seed):
    """Round-robin tournament scored against expected matchup matrix."""
    STRATS = {
        'expansive':  PlacementStrategy.EXPANSIVE,
        'aggressive': PlacementStrategy.AGGRESSIVE,
        'balanced':   PlacementStrategy.BALANCED,
        'turtle':     PlacementStrategy.TURTLE,
    }
    
    matchups = list(combinations(STRATS.keys(), 2))
    results = {}  # (s1, s2) -> win rate for s1
    
    for s1_name, s2_name in matchups:
        s1_wins = 0
        games = 0
        
        for first, second in [(s1_name, s2_name), (s2_name, s1_name)]:
            parts = fresh_game(seed, n_squads=2)
            
            strategies = {
                parts.squads[0].name: STRATS[first],
                parts.squads[1].name: STRATS[second],
            }
            
            tp = TurnProcessor(parts, parts.squads)
            for r in range(12):  # 12 rounds is enough to decide most games
                if tp.game_over:
                    break
                for squad in parts.squads:
                    if not squad.king_alive():
                        continue
                    strat = strategies[squad.name]
                    move = strat.next_move(squad, parts)
                    if move:
                        try:
                            tp.resolve_move(squad, move)
                        except MoveError:
                            pass
                if (r + 1) % 3 == 0:
                    for squad in parts.squads:
                        squad.optimize_facings(parts)
            
            # Who won?
            sq0_alive = parts.squads[0].king_alive()
            sq1_alive = parts.squads[1].king_alive()
            
            if sq0_alive and not sq1_alive:
                winner = first
            elif sq1_alive and not sq0_alive:
                winner = second
            else:
                winner = None  # draw
            
            if winner == s1_name:
                s1_wins += 1
            elif winner is None:
                s1_wins += 0.5  # draw
            games += 1
            
            # Clean up for cache reuse
            for sq in parts.squads:
                for p in sq.pieces:
                    p.place = None
        
        results[(s1_name, s2_name)] = s1_wins / games if games > 0 else 0.5
    
    # Score: how close are actual win rates to targets?
    total_error = 0.0
    for matchup, target in MATCHUP_TARGETS.items():
        actual = results.get(matchup, 0.5)
        total_error += abs(actual - target)
    
    n_matchups = len(MATCHUP_TARGETS)
    avg_error = total_error / n_matchups
    score = max(0.0, 1.0 - avg_error * 4.0)
    
    return score, results



def objective_rps(trial):
    p = apply_params(trial)
    
    scores = {k: [] for k in ['sustain', 'flank', 'queen_1v1', 'king_dies', 'rps']}
    all_matchups = {}
    
    for seed in MAP_SEEDS:
        parts = make_parts(seed)
        parts = fresh_game(seed, n_squads=2)
        
        scores['sustain'].append(score_3pawns_support_queen(parts))
        scores['flank'].append(score_2knights_beat_queen(parts))
        scores['queen_1v1'].append(score_queen_beats_knight(parts))
        scores['king_dies'].append(score_lone_king_dies(parts))
        
        rps_score, matchup_results = score_rps_balance(seed)
        scores['rps'].append(rps_score)
        for k, v in matchup_results.items():
            all_matchups.setdefault(k, []).append(v)
    
    s1 = sum(scores['sustain']) / len(MAP_SEEDS)
    s2 = sum(scores['flank']) / len(MAP_SEEDS)
    s3 = sum(scores['queen_1v1']) / len(MAP_SEEDS)
    s4 = sum(scores['king_dies']) / len(MAP_SEEDS)
    s5 = sum(scores['rps']) / len(MAP_SEEDS)
    
    total = (s1 * 2.0 +   # sustain
             s2 * 2.0 +   # flanking
             s3 * 2.0 +   # queen 1v1
             s4 * 1.0 +   # king pressure
             s5 * 4.0)    # RPS balance — highest weight!
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('king_dies', s4)
    trial.set_user_attr('rps', s5)
    
    for matchup, vals in all_matchups.items():
        label = f"{matchup[0]}_v_{matchup[1]}"
        trial.set_user_attr(label, sum(vals) / len(vals))
    
    return total

# Build initial params from current globals
initial_params = {}
for key, pt in [('pawn', PieceType.PAWN), ('bishop', PieceType.BISHOP),
                ('knight', PieceType.KNIGHT), ('queen', PieceType.QUEEN)]:
    fs = FOOD_TABLE[pt]
    initial_params[f'{key}_diet'] = fs.diet
    initial_params[f'{key}_cost'] = fs.cost
    initial_params[f'{key}_harvest'] = fs.harvest
    initial_params[f'{key}_drain'] = fs.drain
    if key not in ('pawn', 'knight'):  # these have fixed sight
        initial_params[f'{key}_sight'] = fs.bonus_sight
    initial_params[f'{key}_life'] = LIFESPAN[pt]

study_rps = optuna.create_study(direction='maximize', study_name='hex_rps')
study_rps.enqueue_trial(initial_params)
study_rps.optimize(objective_rps, n_trials=40, show_progress_bar=True)

# Results
print(f"\nBest score: {study_rps.best_value:.2f}")
print(f"\nSub-scores:")
for k in ['sustain', 'flank', 'queen_1v1', 'king_dies', 'rps']:
    print(f"  {k}: {study_rps.best_trial.user_attrs[k]:.2f}")
print(f"\nMatchups (row win rate, target in parens):")
for matchup, target in MATCHUP_TARGETS.items():
    label = f"{matchup[0]}_v_{matchup[1]}"
    actual = study_rps.best_trial.user_attrs.get(label, 0.5)
    status = "✓" if abs(actual - target) < 0.15 else "✗"
    print(f"  {matchup[0]:12s} vs {matchup[1]:12s}: {actual:.0%} (target {target:.0%}) {status}")
print(f"\nBest params:")
for k, v in study_rps.best_params.items():
    print(f"  {k}: {v}")
```

In [ ]:
read_url("https://fastht.ml/docs/llms-ctx.txt")

In [ ]:
read_url("https://www.geeksforgeeks.org/python/python-visualize-graphs-generated-in-networkx-using-matplotlibre/")

In [ ]:
!cat ../../HexMagic/game/flag.py

I was wondering about using the networkX drawing code to show sources and sinks and general flow of a squad in a diagram

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a great idea — you've already got `to_nx()` building a rich DiGraph with `harvest`, `diet`, `is_sink`, `is_king` etc. on each node. NetworkX's drawing tools can visualize that directly.

The key pieces you'd want to show:

- **Node color** — green for sources (harvest > diet), red for sinks, gold for king
- **Node size** — proportional to pantry or net food
- **Edge arrows** — sight directions (the flow paths)
- **Labels** — piece name + net food per turn

The simplest starting point would be something like:

```python
G = myStuff.squads[0].to_nx()
pos = nx.spring_layout(G)  # or nx.kamada_kawai_layout(G)
nx.draw(G, pos, with_labels=True)
```

From there you'd layer in color maps and sizing based on the node attributes. A couple of layout choices worth considering:

1. **`spring_layout`** — shows connectivity clustering nicely, but loses spatial meaning
2. **`kamada_kawai_layout`** — similar but tends to spread better
3. **Custom pos from hex coordinates** — use the actual `hex_idx` → grid center positions, so the graph overlays the map spatially

Which approach appeals to you — abstract graph layout, or tied to the actual map positions?

The goal is to combine this with fasthtml and perhaps make it so we could click on it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — you've got two natural paths for this:

1. **Map-integrated** — An overlay like `FoodNetworkOverlay` but with HTMX `hx-get` attrs on each node SVG element, so clicking a piece shows its food stats / connections in a side panel
2. **Abstract graph** — A standalone SVG component using networkx layout positions (not map positions), rendered as an `__ft__` method on Squad, with clickable nodes

Both work well with FastHTML since you're generating SVG server-side already. The key trick is adding `hx-get="/piece/{id}"` and `hx-target="#detail-panel"` to each SVG node element.

Which direction are you leaning — integrated into the map, or a separate diagram panel? And what should clicking a node *do* — show that piece's stats, highlight its sight cone, show its food flow connections?

I think we should have both. I do think the FoodNetworkOverlay could be improved, but lets work on the abstract graph first. Maybe use the avatars for the node pictures?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love it — avatars as graph nodes will look great. Here's what I'm thinking for the approach:

1. Use `nx.kamada_kawai_layout(G)` to get abstract (x, y) positions
2. Scale those positions into an SVG viewBox (say 600×400)
3. Draw directed edges as arrows between nodes
4. At each node position, use `piece.draw_avatar()` (you already have this!) with a colored ring behind it — green for source, red for sink, gold for king
5. Label underneath with name + net food

The `__ft__` method on Squad's graph (or a new class like `SquadFlowDiagram`) would return the SVG wrapped in a `Div`, ready for HTMX click targets later.

Want to start by getting the basic layout + edges rendering, then layer in the avatars? Or jump straight to avatars from the start?

I think a new class would be good. We are only going to want to show the nodes that have been placed on the map and addition to food we should also have how many turns remaining for each node

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. Here's the shape I'd suggest:

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
```

For the node data, each placed piece needs:
- **avatar** (already cached via `fetch_avatar`)
- **net food** = `harvest × tile_tier - diet`
- **turns remaining** = `LIFESPAN[type] - tenure` (king = ∞)
- **role color** — green/red/gold ring

The `__ft__` method would:
1. Build `G = squad.to_nx()` (only placed pieces)
2. Get layout positions via `nx.kamada_kawai_layout(G)`
3. Scale positions into the viewBox
4. Draw edges as arrow paths
5. Draw avatar + ring + label at each node

Does that structure look right, and shall I sketch out the `__ft__` method?

!cat ../../HexMagic/plot/*.py

So we have a HexWrapper class that works with grids and I think we need something similar for the fastai. this would either need to be passed into the function (in which case we couldn't do ft) or as a separate function (which in theory ft could call with a blank or default one_

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking. I'd go with option 2 — `__ft__` renders the static diagram (useful for notebooks, debug), and a separate method returns the interactive FastHTML version:

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    
    def __ft__(self):
        """Static SVG — no interaction."""
        return self._render()
    
    def interactive(self, on_click=None):
        """FastHTML Div with HTMX click targets on nodes."""
        # on_click: (piece) -> dict of hx-* attrs
        return self._render(on_click=on_click)
```

The `on_click` callback would mirror `HexWrapper.route` but for pieces:

```python
# Usage:
def piece_click(piece):
    return {"hx-get": f"/piece/{piece.id}", "hx-target": "#detail-panel"}

show(diagram.interactive(on_click=piece_click))
```

Does that pattern work for you? Then we can flesh out `_render` with the nx layout + avatars + edges.

that pattern works well. Do we type this at all using the __future__ 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

On Python 3.12 you don't need `from __future__ import annotations` — PEP 604 union syntax (`X | Y`) and built-in generics (`list[int]`, `dict[str, int]`) all work natively.

For typing the callback, you'd use `Callable` from `collections.abc`:

```python
from collections.abc import Callable

@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    on_click: Callable[[Piece], dict] | None = None
```

That way `on_click` is part of the dataclass rather than a method arg — cleaner for `__ft__` since it doesn't need to be passed separately. `interactive()` could then just be a factory that sets `on_click`:

```python
diagram = SquadFlowDiagram(squad, parts, on_click=piece_click)
show(diagram)  # __ft__ uses on_click if present
```

Want to go that route, or keep `on_click` as a method parameter?

lets go this route

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Heads up: the context is getting large, so some earlier messages have been trimmed from my view. If I miss something you defined earlier, just point me to it.

Here's the skeleton:

```python
from collections.abc import Callable

@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _render(self) -> Div:
        G = self.squad.to_nx()
        if len(G) < 2:
            return Div(P(f"{self.squad.name}: not enough placed pieces"))
        
        # 1. Layout
        pos = nx.kamada_kawai_layout(G)
        
        # 2. Scale positions into SVG viewBox
        # 3. Build SVG: edges → node rings → avatars → labels
        # 4. If on_click, wrap each node group in an element with hx-* attrs
        
        pass  # we'll fill this in
```

Shall I write out step 2 (the coordinate scaling) next, or jump to the full `_render` implementation?

How much of this can you write out?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I can write the whole thing. Here goes:

```python
from collections.abc import Callable

@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    avatar_size: float = 48
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _render(self):
        G = self.squad.to_nx()
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"), cls=TextPresets.muted_sm)
        
        flag = self.squad.flag
        pad = self.avatar_size * 0.8
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        # Scale to SVG coords
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            sx = pad + (nx_pos[0] - x_min) / x_range * inner_w
            sy = pad + (nx_pos[1] - y_min) / y_range * inner_h
            return sx, sy
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Edge arrows
        edge_parts = []
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        marker = (
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.5"/>'
            f'</marker></defs>'
        )
        edge_parts.append(marker)
        
        r = self.avatar_size / 2
        for u, v in G.edges():
            x1, y1 = svg_pos[u]
            x2, y2 = svg_pos[v]
            # Shorten to avoid overlapping avatars
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1:
                continue
            ux, uy = dx / dist, dy / dist
            x1s = x1 + ux * (r + 4)
            y1s = y1 + uy * (r + 4)
            x2s = x2 - ux * (r + 4)
            y2s = y2 - uy * (r + 4)
            edge_parts.append(
                f'<line x1="{x1s:.1f}" y1="{y1s:.1f}" x2="{x2s:.1f}" y2="{y2s:.1f}" '
                f'stroke="{flag.darkPrimary}" stroke-opacity="0.35" stroke-width="1.5" '
                f'marker-end="url(#{arrow_id})" stroke-dasharray="4,3"/>'
            )
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # Nodes: ring + avatar + label
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            harvest = data['harvest']
            diet = data['diet']
            net = harvest - diet
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1  # -1 = immortal
            
            # Ring color: green=source, red=sink, gold=king
            if data['is_king']:
                ring_color = "#FFD700"
                ring_width = 3
            elif net > 0:
                ring_color = "#4CAF50"
                ring_width = 2.5
            else:
                ring_color = "#E53935"
                ring_width = 2.5
            
            # Optional HTMX wrapper
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            # Ring
            ring_r = r + 4
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" stroke="{ring_color}" stroke-width="{ring_width}"/>'
            )
            
            # Avatar
            piece.register_avatar(builder)
            aid = piece.avatar_id()
            ax = cx - r
            ay = cy - r
            avatar = (
                f'<use href="#{aid}" x="{ax:.1f}" y="{ay:.1f}" '
                f'width="{self.avatar_size:.0f}" height="{self.avatar_size:.0f}"/>'
            )
            
            # Piece type badge (top-right)
            badge = (
                f'<text x="{cx + r - 2:.1f}" y="{cy - r + 6:.1f}" '
                f'text-anchor="middle" font-size="12" '
                f'dominant-baseline="central">{piece.type.icon}</text>'
            )
            
            # Name label (below)
            name_y = cy + r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>'
            )
            
            # Food + turns label (below name)
            info_y = name_y + 13
            food_color = "#4CAF50" if net >= 0 else "#E53935"
            turns_str = f"∞" if turns_left < 0 else f"{turns_left}t"
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{food_color}">{net:+d}🌾</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>'
            )
            
            node_parts.append(
                f'{group_open}{ring}{avatar}{badge}{name_label}{info_label}</g>'
            )
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # Summary footer
        placed = [p for p in self.squad.pieces if p.place is not None]
        total_harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        total_diet = sum(p.food.diet for p in placed)
        king = next((p for p in placed if p.type == PieceType.KING), None)
        king_pantry = king.place.pantry if king else 0
        
        G_undir = G.to_undirected()
        bottlenecks = len(list(nx.articulation_points(G_undir))) if len(G) > 1 else 0
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        
        footer = P(
            f"🌾 {total_harvest - total_diet:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"⚠️ {bottlenecks} bottleneck{'s' if bottlenecks != 1 else ''} · "
            f"🎯 {target_str}",
            cls=TextPresets.muted_sm
        )
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=(H4(self.squad.name), 
                    P(f"{self.squad.flag.name} · {len(placed)} placed", 
                      cls=TextPresets.muted_sm)),
        )
```

Usage:

```python
# Static (notebook)
diagram = SquadFlowDiagram(myStuff.squads[0], myStuff)
show(diagram)

# Interactive (FastHTML app)
def on_piece_click(piece):
    return {"hx-get": f"/piece/{piece.id}", "hx-target": "#detail-panel", "hx-swap": "innerHTML"}

diagram = SquadFlowDiagram(myStuff.squads[0], myStuff, on_click=on_piece_click)
show(diagram)
```

Want me to also write the `/piece/{id}` route handler that the click targets would hit?

Can we test the visualization? I will do the callbacks later

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Sure! Let's get some pieces placed and test it:

```python
# Make sure we have a game with placed pieces
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

# Place a few pawns per squad so the graph has edges
for _ in range(3):
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            tp.resolve_move(squad, move)

# Test the diagram
diagram = SquadFlowDiagram(myStuff.squads[0], myStuff)
show(diagram)
```

Drop that in a code cell and see how it looks. If the avatars don't render or the layout is off, we'll iterate from there.

In [ ]:
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9       # chess piece scale factor
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _compute_flow(self, G):
        """Run max-flow and return dict of (u,v) -> flow amount."""
        flows = {}
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            pantry = data['piece'].place.pantry
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        flows[(u, v)] = amt
        return flows
    
    def _render(self):
        G = self.squad.to_nx()
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale  # half-size of chess piece
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Compute flow ---
        flows = self._compute_flow(G) if len(G) > 1 else {}
        max_flow = max(flows.values(), default=1)
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges with flow ---
        edge_parts = []
        shrink = node_r + 8
        
        # Draw all sight edges (thin dashed)
        seen = set()
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            x1, y1 = svg_pos[u]
            x2, y2 = svg_pos[v]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            # Check if there's flow on this edge (either direction)
            fwd = flows.get((u, v), 0)
            bwd = flows.get((v, u), 0)
            total = fwd + bwd
            
            if total > 0:
                # Flow edge: solid, thickness proportional to flow
                w = 1.5 + (total / max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                # Sight-only edge: thin dashed
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            # Flow label at midpoint
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                # Offset perpendicular to avoid overlap with line
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:+.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes: ring + chess piece + labels ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            harvest = data['harvest']
            diet = data['diet']
            net = harvest - diet
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            
            # Ring color
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif net > 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            # White circle background + colored ring
            ring_r = node_r + 6
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece SVG
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name label
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Food + turns label
            info_y = name_y + 13
            food_color = "#4CAF50" if net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            pantry = piece.place.pantry if piece.place else 0
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{food_color}">{net:+d}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        placed = [p for p in self.squad.pieces if p.place is not None]
        total_harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        total_diet = sum(p.food.diet for p in placed)
        king = next((p for p in placed if p.type == PieceType.KING), None)
        king_pantry = king.place.pantry if king else 0
        
        G_undir = G.to_undirected()
        bottlenecks = len(list(nx.articulation_points(G_undir))) if len(G) > 1 else 0
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        
        total_flow = sum(flows.values())
        
        footer = P(
            f"🌾 {total_harvest - total_diet:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {total_flow:.0f} flow · "
            f"⚠️ {bottlenecks} bottleneck{'s' if bottlenecks != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {len(placed)} placed")),
            ),
        )


In [ ]:
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _compute_flow(self, G):
        """Run max-flow and return per-node net flow + edge flows."""
        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}
        
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        edge_flows[(u, v)] = amt
                        node_outflow[u] += amt
                        node_inflow[v] += amt
        
        return edge_flows, node_inflow, node_outflow
    
    def _render(self):
        G = self.squad.to_nx()
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Compute flow ---
        if len(G) > 1:
            edge_flows, node_inflow, node_outflow = self._compute_flow(G)
        else:
            edge_flows, node_inflow, node_outflow = {}, {n: 0 for n in G.nodes}, {n: 0 for n in G.nodes}
        max_flow = max(edge_flows.values(), default=1)
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges ---
        edge_parts = []
        shrink = node_r + 8
        seen = set()
        
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            
            fwd = edge_flows.get((u, v), 0)
            bwd = edge_flows.get((v, u), 0)
            net_flow = fwd - bwd
            total = abs(net_flow)
            
            if net_flow >= 0:
                src, dst = u, v
            else:
                src, dst = v, u
            
            x1, y1 = svg_pos[src]
            x2, y2 = svg_pos[dst]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            if total > 0:
                w = 1.5 + (total / max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            raw_net = data['harvest'] - data['diet']
            post_net = raw_net + node_inflow.get(nid, 0) - node_outflow.get(nid, 0)
            
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            pantry = piece.place.pantry if piece.place else 0
            
            # Ring color based on post-flow net
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif post_net >= 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            # White circle + ring
            ring_r = node_r + 6
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Post-flow net + pantry + turns
            info_y = name_y + 13
            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        placed = [p for p in self.squad.pieces if p.place is not None]
        total_harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        total_diet = sum(p.food.diet for p in placed)
        king = next((p for p in placed if p.type == PieceType.KING), None)
        king_pantry = king.place.pantry if king else 0
        
        G_undir = G.to_undirected()
        bottlenecks = len(list(nx.articulation_points(G_undir))) if len(G) > 1 else 0
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        total_flow = sum(edge_flows.values())
        
        footer = P(
            f"🌾 {total_harvest - total_diet:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {total_flow:.0f} flow · "
            f"⚠️ {bottlenecks} bottleneck{'s' if bottlenecks != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {len(placed)} placed")),
            ),
        )


In [ ]:
# Make sure we have a game with placed pieces
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

# Place a few pawns per squad so the graph has edges
for _ in range(3):
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            tp.resolve_move(squad, move)

# Test the diagram
diagram = SquadFlowDiagram(myStuff.squads[0], myStuff)
show(diagram)


This is great. Could we make the piece icons slight larger or use the svg from CountryFlag. Is there away we could represent the amount of food that is being sent?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — two improvements:

**1. Chess piece SVGs instead of avatars** — use `flag.piece_svg()` which already renders flag-themed pieces at any scale. Bigger and more readable.

**2. Food flow on edges** — run the max-flow computation and label each edge with the amount transferred. Thicker edges = more flow.

Here's the updated `_render`:

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9       # chess piece scale factor
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _compute_flow(self, G):
        """Run max-flow and return dict of (u,v) -> flow amount."""
        flows = {}
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            pantry = data['piece'].place.pantry
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        flows[(u, v)] = amt
        return flows
    
    def _render(self):
        G = self.squad.to_nx()
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale  # half-size of chess piece
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Compute flow ---
        flows = self._compute_flow(G) if len(G) > 1 else {}
        max_flow = max(flows.values(), default=1)
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges with flow ---
        edge_parts = []
        shrink = node_r + 8
        
        # Draw all sight edges (thin dashed)
        seen = set()
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            x1, y1 = svg_pos[u]
            x2, y2 = svg_pos[v]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            # Check if there's flow on this edge (either direction)
            fwd = flows.get((u, v), 0)
            bwd = flows.get((v, u), 0)
            total = fwd + bwd
            
            if total > 0:
                # Flow edge: solid, thickness proportional to flow
                w = 1.5 + (total / max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                # Sight-only edge: thin dashed
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            # Flow label at midpoint
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                # Offset perpendicular to avoid overlap with line
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:+.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes: ring + chess piece + labels ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            harvest = data['harvest']
            diet = data['diet']
            net = harvest - diet
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            
            # Ring color
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif net > 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            # White circle background + colored ring
            ring_r = node_r + 6
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece SVG
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name label
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Food + turns label
            info_y = name_y + 13
            food_color = "#4CAF50" if net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            pantry = piece.place.pantry if piece.place else 0
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{food_color}">{net:+d}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        placed = [p for p in self.squad.pieces if p.place is not None]
        total_harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        total_diet = sum(p.food.diet for p in placed)
        king = next((p for p in placed if p.type == PieceType.KING), None)
        king_pantry = king.place.pantry if king else 0
        
        G_undir = G.to_undirected()
        bottlenecks = len(list(nx.articulation_points(G_undir))) if len(G) > 1 else 0
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        
        total_flow = sum(flows.values())
        
        footer = P(
            f"🌾 {total_harvest - total_diet:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {total_flow:.0f} flow · "
            f"⚠️ {bottlenecks} bottleneck{'s' if bottlenecks != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {len(placed)} placed")),
            ),
        )
```

Changes from v1:
- **Chess piece SVGs** via `flag.piece_svg()` at `node_scale=0.9` — bigger and flag-themed
- **Flow computation** via `_compute_flow` — runs max-flow, returns `(u,v) → amount`
- **Edge thickness** proportional to flow amount, green for active flow, gray dashed for sight-only
- **Flow labels** at edge midpoints showing `+N🌾`
- **Pantry shown** per node (📦) alongside net food and turns remaining
- **Total flow** in footer summary

Test it with:

```python
diagram = SquadFlowDiagram(myStuff.squads[0], myStuff)
show(diagram)
```

Shouldn't the pawns be flowing to the king?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right! The flow direction is wrong. The issue is in the edge drawing — it deduplicates with `sorted([u, v])` so it picks an arbitrary direction for the arrow. The arrow should point in the direction of **net flow** (sources → sinks, i.e. pawns → king).

Replace the edge drawing section in `_render` with:

```python
        # --- Edges with flow ---
        edge_parts = []
        shrink = node_r + 8
        
        seen = set()
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            
            # Net flow direction
            fwd = flows.get((u, v), 0)
            bwd = flows.get((v, u), 0)
            net_flow = fwd - bwd
            total = abs(net_flow)
            
            # Draw arrow in direction of net flow (source → sink)
            if net_flow >= 0:
                src, dst = u, v
            else:
                src, dst = v, u
            
            x1, y1 = svg_pos[src]
            x2, y2 = svg_pos[dst]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            if total > 0:
                w = 1.5 + (total / max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
```

The key fix: `net_flow = fwd - bwd` determines direction. If positive, `u→v`; if negative, flip to `v→u`. So the green arrows will now point from pawns (surplus) toward the king (sink). The flow labels also drop the `+` prefix since they just show magnitude now.

So I am wondering about sources and sinks outside of the graph. is the -2 for brandon mean he is losing 2 food and the three aren't supporting him?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question — the `-2🌾` on Brandon is his **raw** production rate: `harvest × tile_tier - diet = 1.0 × 0 - 2 = -2`. He's on a tier 0 hex, so he earns nothing and pays 2 diet.

The flow arrows **should** be covering that deficit. The pawns each produce +7 surplus, and the max-flow routes 2 of that to the king. So in practice Brandon isn't starving — his pantry stays healthy (📦85) because the network feeds him.

But this reveals a UX gap in the diagram: we're showing two different things that could be confused:

1. **Node label** (`-2🌾`) = raw production rate (before flow sharing)
2. **Edge labels** (`+12🌾`) = flow being transferred

It would be clearer if we also showed the **post-flow** net on each node. Something like:

```
Brandon (♚)
-2🌾 raw → +0🌾 net
📦85 · ∞
```

Or even simpler — show **both** numbers on the node:

```
⚡-2  →  ✅+0
```

Where ⚡ is the raw production and ✅ is after receiving flow.

We could compute this from the flow dict — for each node, `post_flow_net = raw_net + inflow - outflow`. Want me to add that to the node labels?

I think it is ok to just show the net. There is also a more complicated version of this where we show potential pieces and how they would affect the diagram, but we can just work on getting the info better. Can you rewrite the function?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _compute_flow(self, G):
        """Run max-flow and return per-node net flow + edge flows."""
        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}
        
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        edge_flows[(u, v)] = amt
                        node_outflow[u] += amt
                        node_inflow[v] += amt
        
        return edge_flows, node_inflow, node_outflow
    
    def _render(self):
        G = self.squad.to_nx()
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Compute flow ---
        if len(G) > 1:
            edge_flows, node_inflow, node_outflow = self._compute_flow(G)
        else:
            edge_flows, node_inflow, node_outflow = {}, {n: 0 for n in G.nodes}, {n: 0 for n in G.nodes}
        max_flow = max(edge_flows.values(), default=1)
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges ---
        edge_parts = []
        shrink = node_r + 8
        seen = set()
        
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            
            fwd = edge_flows.get((u, v), 0)
            bwd = edge_flows.get((v, u), 0)
            net_flow = fwd - bwd
            total = abs(net_flow)
            
            if net_flow >= 0:
                src, dst = u, v
            else:
                src, dst = v, u
            
            x1, y1 = svg_pos[src]
            x2, y2 = svg_pos[dst]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            if total > 0:
                w = 1.5 + (total / max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            raw_net = data['harvest'] - data['diet']
            post_net = raw_net + node_inflow.get(nid, 0) - node_outflow.get(nid, 0)
            
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            pantry = piece.place.pantry if piece.place else 0
            
            # Ring color based on post-flow net
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif post_net >= 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            # White circle + ring
            ring_r = node_r + 6
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Post-flow net + pantry + turns
            info_y = name_y + 13
            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        placed = [p for p in self.squad.pieces if p.place is not None]
        total_harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        total_diet = sum(p.food.diet for p in placed)
        king = next((p for p in placed if p.type == PieceType.KING), None)
        king_pantry = king.place.pantry if king else 0
        
        G_undir = G.to_undirected()
        bottlenecks = len(list(nx.articulation_points(G_undir))) if len(G) > 1 else 0
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        total_flow = sum(edge_flows.values())
        
        footer = P(
            f"🌾 {total_harvest - total_diet:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {total_flow:.0f} flow · "
            f"⚠️ {bottlenecks} bottleneck{'s' if bottlenecks != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {len(placed)} placed")),
            ),
        )
```

Key change: `_compute_flow` now returns `node_inflow` and `node_outflow` dicts alongside edge flows. Each node's label shows `post_net = raw_net + inflow - outflow`, so the king should show `+0🌾` (deficit covered) and pawns show whatever surplus they keep after donating. The ring color is also based on post-flow net now.

In [ ]:
def FoodNetworkOverlay(
    focus_squad: Squad | None = None,
    edge_opacity: float = 0.6,
    bottleneck_r_mult: float = 1.4,
    **kw
) -> OverlaySpec:
    """Food sharing network with directed edges, flow weighting, target intent, and pantry bars."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        squads_to_draw = [focus_squad] if focus_squad else ctx.squads
        
        for squad in ctx.squads:
            G = squad.to_nx()
            if len(G) < 1:
                continue
            
            flag = squad.flag
            glyphs = DiagramGlyphs(flag, size=grid.radius * 0.5)
            glyphs.register_styles(ctx.builder)
            is_focused = focus_squad is None or squad is focus_squad
            
            # --- Edge styles ---
            if is_focused and len(G) > 1:
                U = G.to_undirected()
                bridges = set(nx.bridges(U))
                
                # Arrow marker
                arrow_id = f"fna_{flag.name.replace(' ', '_')}"
                ctx.builder.add_definition(SVGDef("marker", arrow_id,
                    f'<path d="M 0 0 L 8 4 L 0 8 z" fill="{flag.darkPrimary}" opacity="0.6"/>',
                    viewBox="0 0 8 8", refX="8", refY="4",
                    markerWidth="5", markerHeight="5", orient="auto-start-reverse"))
                
                for u, v in G.edges():
                    u_idx = G.nodes[u]['hex_idx']
                    v_idx = G.nodes[v]['hex_idx']
                    
                    # Edge criticality: bridge = thick + solid, redundant = thin + dashed
                    edge_key = (min(u, v), max(u, v))
                    is_bridge = (u, v) in bridges or (v, u) in bridges
                    sw = max(2.5, grid.radius * 0.1) if is_bridge else max(1.2, grid.radius * 0.05)
                    dash = "" if is_bridge else 'stroke-dasharray="5,3"'
                    opacity = edge_opacity if is_bridge else edge_opacity * 0.5
                    
                    for fi_u in ctx.fine_indices(u_idx):
                        for fi_v in ctx.fine_indices(v_idx):
                            if 0 <= fi_u < N and 0 <= fi_v < N:
                                c1 = grid.hexes[fi_u].center
                                c2 = grid.hexes[fi_v].center
                                
                                dx = c2.x - c1.x
                                dy = c2.y - c1.y
                                dist = math.sqrt(dx*dx + dy*dy)
                                if dist < 1:
                                    continue
                                ux, uy = dx / dist, dy / dist
                                shrink = grid.radius * 0.4
                                x1 = c1.x + ux * shrink
                                y1 = c1.y + uy * shrink
                                x2 = c2.x - ux * shrink
                                y2 = c2.y - uy * shrink
                                
                                parts.append(
                                    f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                                    f'x2="{x2:.1f}" y2="{y2:.1f}" '
                                    f'stroke="{flag.darkPrimary}" stroke-opacity="{opacity:.2f}" '
                                    f'stroke-width="{sw:.1f}" stroke-linecap="round" '
                                    f'marker-end="url(#{arrow_id})" {dash}/>')
            
            # --- Target direction indicator ---
            if is_focused:
                target = getattr(squad, '_target', None)
                if target and target.hex_idx >= 0:
                    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
                    if king_id:
                        k_idx = G.nodes[king_id]['hex_idx']
                        t_idx = target.hex_idx
                        for fi_k in ctx.fine_indices(k_idx):
                            if 0 <= fi_k < N:
                                kc = grid.hexes[fi_k].center
                                # Find direction toward target
                                t_hp = squad.pieces[0].place.parent.i2hp(t_idx)
                                k_hp = squad.pieces[0].place.parent.i2hp(k_idx)
                                # Use fine grid center if target is in view
                                for fi_t in ctx.fine_indices(t_idx):
                                    if 0 <= fi_t < N:
                                        tc = grid.hexes[fi_t].center
                                        dx = tc.x - kc.x
                                        dy = tc.y - kc.y
                                        dist = math.sqrt(dx*dx + dy*dy)
                                        if dist < 1:
                                            break
                                        ux, uy = dx / dist, dy / dist
                                        arrow_len = grid.radius * 2.5
                                        ax = kc.x + ux * arrow_len
                                        ay = kc.y + uy * arrow_len
                                        
                                        # Target type icon
                                        icon = "⚔️" if target.type == TargetType.KING else "🌾"
                                        
                                        parts.append(
                                            f'<line x1="{kc.x:.1f}" y1="{kc.y:.1f}" '
                                            f'x2="{ax:.1f}" y2="{ay:.1f}" '
                                            f'stroke="{flag.primary}" stroke-opacity="0.4" '
                                            f'stroke-width="{max(3, grid.radius * 0.15):.1f}" '
                                            f'stroke-linecap="round" stroke-dasharray="8,4"/>')
                                        parts.append(
                                            f'<text x="{ax:.1f}" y="{ay:.1f}" '
                                            f'text-anchor="middle" font-size="{max(12, grid.radius * 0.5):.0f}" '
                                            f'dominant-baseline="central">{icon}</text>')
                                        break
            
            # --- Nodes ---
            bottlenecks = set()
            if len(G) > 1:
                bottlenecks = set(nx.articulation_points(G.to_undirected()))
            
            for nid, data in G.nodes(data=True):
                piece = data['piece']
                harvest = data['harvest']
                diet = data['diet']
                net = harvest - diet
                idx = data['hex_idx']
                is_bottleneck = nid in bottlenecks
                
                tenure = piece.place.tenure if piece.place else 0
                life = LIFESPAN.get(piece.type, 999)
                turns_left = max(0, life - tenure) if life < 999 else -1
                
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        c = grid.hexes[fi].center
                        r = grid.radius * 0.25
                        
                        # Bottleneck warning ring
                        if is_bottleneck and is_focused:
                            parts.append(glyphs.ring(
                                c, r * bottleneck_r_mult,
                                glyphs.danger_style()))
                        
                        # Node glyph (always shown)
                        if net > 0:
                            parts.append(glyphs.sheaf(c, size=r * 2.5))
                        elif net < 0:
                            parts.append(glyphs.shield(c, size=r * 2))
                        else:
                            parts.append(glyphs.ring(c, r))
                        
                        if is_focused:
                            # --- Pantry bar (#6) ---
                            pantry = piece.place.pantry if piece.place else 0
                            bar_w = grid.radius * 0.8
                            bar_h = max(3, grid.radius * 0.1)
                            bar_x = c.x - bar_w / 2
                            bar_y = c.y + r * 2.0
                            
                            # Background
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="#ddd" opacity="0.6"/>')
                            
                            # Fill (clamp 0-1, based on pantry relative to cost*2 as "full")
                            max_pantry = max(piece.food.cost * 2, 20)
                            fill_frac = max(0, min(1, pantry / max_pantry))
                            fill_color = "#4CAF50" if pantry > 0 else "#E53935"
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w * fill_frac:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="{fill_color}" opacity="0.8"/>')
                            
                            # --- Net + turns label ---
                            label_y = bar_y + bar_h + max(8, grid.radius * 0.3)
                            food_color = "#4CAF50" if net >= 0 else "#E53935"
                            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
                            fs = max(8, grid.radius * 0.28)
                            
                            parts.append(
                                f'<text x="{c.x:.1f}" y="{label_y:.1f}" '
                                f'text-anchor="middle" font-size="{fs:.0f}">'
                                f'<tspan fill="{food_color}" font-weight="bold">{net:+d}</tspan>'
                                f' <tspan fill="#888">{turns_str}</tspan>'
                                f'</text>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)


should/could FoodNetworkOverlay use SquadFlowDiagram to help draw its edges?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

They could share the **computation** but not really the **rendering** — the two draw in fundamentally different coordinate systems (hex grid centers vs abstract layout positions).

The cleanest refactor would be to extract the shared flow math into a standalone helper:

```python
def compute_squad_flow(G) -> tuple[dict, dict, dict]:
    """Compute max-flow for a squad's sight graph.
    
    Returns:
        edge_flows: dict[(u,v), float] — flow on each directed edge
        node_inflow: dict[nid, float] — total flow into each node
        node_outflow: dict[nid, float] — total flow out of each node
    """
    edge_flows = {}
    node_inflow = {n: 0 for n in G.nodes}
    node_outflow = {n: 0 for n in G.nodes}
    
    F = nx.DiGraph()
    F.add_node('S')
    F.add_node('T')
    for nid, data in G.nodes(data=True):
        net = data['harvest'] - data['diet']
        if net > 0:
            F.add_edge('S', nid, capacity=net)
        elif net < 0:
            F.add_edge(nid, 'T', capacity=-net)
    for u, v in G.edges():
        F.add_edge(u, v, capacity=999)
        F.add_edge(v, u, capacity=999)
    
    if F.out_degree('S') > 0 and F.in_degree('T') > 0:
        _, flow_dict = nx.maximum_flow(F, 'S', 'T')
        for u in G.nodes:
            for v in G.nodes:
                if u == v: continue
                amt = flow_dict.get(u, {}).get(v, 0)
                if amt > 0:
                    edge_flows[(u, v)] = amt
                    node_outflow[u] += amt
                    node_inflow[v] += amt
    
    return edge_flows, node_inflow, node_outflow
```

Then both consumers become:

```python
# In SquadFlowDiagram._render:
edge_flows, node_inflow, node_outflow = compute_squad_flow(G)

# In FoodNetworkOverlay render():
edge_flows, node_inflow, node_outflow = compute_squad_flow(G)
```

And `FoodNetworkOverlay` could use the edge flows to:
- **Weight map edge thickness** by flow amount (thicker = more food moving)
- **Color edges** green for active flow, gray dashed for sight-only
- Show **post-flow net** on the map labels instead of raw net

But the actual SVG line drawing stays separate — one uses `grid.hexes[fi].center` coordinates, the other uses `kamada_kawai_layout` positions. Trying to share that rendering code would make both harder to read.

So: **share `compute_squad_flow`, keep rendering separate**. Want me to refactor both to use the shared helper?

Should we have a SquadFlow Class?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — a data class that owns the graph + flow computation, consumed by both renderers:

```python
@dataclass
class SquadFlow:
    """Computed food flow state for a squad. Shared by diagram + overlay renderers."""
    squad: Squad
    G: nx.DiGraph
    edge_flows: dict  # (u, v) -> float
    node_inflow: dict  # nid -> float
    node_outflow: dict  # nid -> float
    
    @classmethod
    def from_squad(cls, squad: Squad) -> 'SquadFlow':
        G = squad.to_nx()
        if len(G) < 2:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)
        
        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}
        
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        edge_flows[(u, v)] = amt
                        node_outflow[u] += amt
                        node_inflow[v] += amt
        
        return cls(squad, G, edge_flows, node_inflow, node_outflow)
    
    def post_net(self, nid) -> float:
        """Net food for a node after flow redistribution."""
        data = self.G.nodes[nid]
        return data['harvest'] - data['diet'] + self.node_inflow[nid] - self.node_outflow[nid]
    
    def edge_flow(self, u, v) -> float:
        """Net flow on undirected edge (positive = u→v)."""
        return self.edge_flows.get((u, v), 0) - self.edge_flows.get((v, u), 0)
    
    @property
    def max_flow(self) -> float:
        return max(self.edge_flows.values(), default=1)
    
    @property
    def total_flow(self) -> float:
        return sum(self.edge_flows.values())
    
    @property
    def bottlenecks(self) -> set:
        if len(self.G) < 2: return set()
        return set(nx.articulation_points(self.G.to_undirected()))
    
    @property 
    def total_surplus(self) -> int:
        placed = [p for p in self.squad.pieces if p.place is not None]
        return sum(int(p.food.harvest * p.place.harvest) for p in placed) - sum(p.food.diet for p in placed)
```

Then the consumers simplify to:

```python
# SquadFlowDiagram
class SquadFlowDiagram:
    ...
    def _render(self):
        flow = SquadFlow.from_squad(self.squad)
        # use flow.G, flow.edge_flow(u,v), flow.post_net(nid), flow.bottlenecks, etc.

# FoodNetworkOverlay
def render(ctx):
    for squad in ctx.squads:
        flow = SquadFlow.from_squad(squad)
        # use flow.edge_flows for edge thickness, flow.post_net(nid) for labels
```

The nice thing is `SquadFlow` is also useful outside rendering — strategy code could inspect `flow.bottlenecks` or `flow.total_surplus` to make decisions. And it's cheap to cache if needed (`squad._flow = SquadFlow.from_squad(squad)` once per turn).

@dataclass
class SquadFlow:
    """Computed food flow state for a squad. Shared by diagram + overlay renderers."""
    squad: Squad
    G: nx.DiGraph
    edge_flows: dict  # (u, v) -> float
    node_inflow: dict  # nid -> float
    node_outflow: dict  # nid -> float
    
    @classmethod
    def from_squad(cls, squad: Squad) -> 'SquadFlow':
        G = squad.to_nx()
        if len(G) < 2:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)
        
        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}
        
        F = nx.DiGraph()
        F.add_node('S')
        F.add_node('T')
        for nid, data in G.nodes(data=True):
            net = data['harvest'] - data['diet']
            if net > 0:
                F.add_edge('S', nid, capacity=net)
            elif net < 0:
                F.add_edge(nid, 'T', capacity=-net)
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            _, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for u in G.nodes:
                for v in G.nodes:
                    if u == v: continue
                    amt = flow_dict.get(u, {}).get(v, 0)
                    if amt > 0:
                        edge_flows[(u, v)] = amt
                        node_outflow[u] += amt
                        node_inflow[v] += amt
        
        return cls(squad, G, edge_flows, node_inflow, node_outflow)
    
    def post_net(self, nid) -> float:
        """Net food for a node after flow redistribution."""
        data = self.G.nodes[nid]
        return data['harvest'] - data['diet'] + self.node_inflow[nid] - self.node_outflow[nid]
    
    def edge_flow(self, u, v) -> float:
        """Net flow on undirected edge (positive = u→v)."""
        return self.edge_flows.get((u, v), 0) - self.edge_flows.get((v, u), 0)
    
    @property
    def max_flow(self) -> float:
        return max(self.edge_flows.values(), default=1)
    
    @property
    def total_flow(self) -> float:
        return sum(self.edge_flows.values())
    
    @property
    def bottlenecks(self) -> set:
        if len(self.G) < 2: return set()
        return set(nx.articulation_points(self.G.to_undirected()))
    
    @property 
    def total_surplus(self) -> int:
        placed = [p for p in self.squad.pieces if p.place is not None]
        return sum(int(p.food.harvest * p.place.harvest) for p in placed) - sum(p.food.diet for p in placed)


can you give me the updated SquadFlowDiagram using SquadFlow

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _render(self):
        flow = SquadFlow.from_squad(self.squad)
        G = flow.G
        
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges ---
        edge_parts = []
        shrink = node_r + 8
        seen = set()
        
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            
            net_flow = flow.edge_flow(u, v)
            total = abs(net_flow)
            
            # Arrow points in direction of net flow
            if net_flow >= 0:
                src, dst = u, v
            else:
                src, dst = v, u
            
            x1, y1 = svg_pos[src]
            x2, y2 = svg_pos[dst]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            if total > 0:
                w = 1.5 + (total / flow.max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            # Bridge highlight
            is_bridge = (u in flow.bottlenecks or v in flow.bottlenecks)
            if is_bridge and total == 0:
                color = "#E53935"
                opacity = 0.4
                w = 1.5
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            post_net = flow.post_net(nid)
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            pantry = piece.place.pantry if piece.place else 0
            
            # Ring color based on post-flow net
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif post_net >= 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # Bottleneck warning
            is_bn = nid in flow.bottlenecks
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            ring_r = node_r + 6
            
            # Bottleneck outer ring
            bn_ring = ""
            if is_bn:
                bn_ring = (
                    f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r + 4:.1f}" '
                    f'fill="none" stroke="#E53935" stroke-width="2" '
                    f'stroke-dasharray="4,2" opacity="0.7"/>')
            
            # White background + colored ring
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Post-flow net + pantry + turns
            info_y = name_y + 13
            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{bn_ring}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        king = next((p for p in self.squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        king_pantry = king.place.pantry if king else 0
        placed_count = sum(1 for p in self.squad.pieces if p.place is not None)
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        
        bn_count = len(flow.bottlenecks)
        
        footer = P(
            f"🌾 {flow.total_surplus:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {flow.total_flow:.0f} flow · "
            f"⚠️ {bn_count} bottleneck{'s' if bn_count != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {placed_count} placed")),
            ),
        )
```

Changes from the previous version:
- **`SquadFlow.from_squad()`** replaces inline `_compute_flow` — single source of truth
- **`flow.edge_flow(u, v)`** gives net directed flow per edge
- **`flow.post_net(nid)`** gives post-redistribution net per node
- **`flow.bottlenecks`** used for both dashed red outer rings on nodes and edge highlighting
- **`flow.total_surplus`** and **`flow.total_flow`** in footer

And so what would the FoodNetworkOverlay look like now?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
def FoodNetworkOverlay(
    focus_squad: Squad | None = None,
    edge_opacity: float = 0.6,
    bottleneck_r_mult: float = 1.4,
    **kw
) -> OverlaySpec:
    """Food sharing network on the map, powered by SquadFlow."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            flow = SquadFlow.from_squad(squad)
            G = flow.G
            if len(G) < 1: continue
            
            flag = squad.flag
            glyphs = DiagramGlyphs(flag, size=grid.radius * 0.5)
            glyphs.register_styles(ctx.builder)
            is_focused = focus_squad is None or squad is focus_squad
            
            # --- Edges ---
            if is_focused and len(G) > 1:
                U = G.to_undirected()
                bridges = set(nx.bridges(U))
                
                arrow_id = f"fna_{flag.name.replace(' ', '_')}"
                ctx.builder.add_definition(SVGDef("marker", arrow_id,
                    f'<path d="M 0 0 L 8 4 L 0 8 z" fill="{flag.darkPrimary}" opacity="0.6"/>',
                    viewBox="0 0 8 8", refX="8", refY="4",
                    markerWidth="5", markerHeight="5", orient="auto-start-reverse"))
                
                seen = set()
                for u, v in G.edges():
                    key = tuple(sorted([u, v]))
                    if key in seen: continue
                    seen.add(key)
                    
                    u_idx = G.nodes[u]['hex_idx']
                    v_idx = G.nodes[v]['hex_idx']
                    
                    # Flow direction
                    net_flow = flow.edge_flow(u, v)
                    total = abs(net_flow)
                    if net_flow >= 0:
                        src_idx, dst_idx = u_idx, v_idx
                    else:
                        src_idx, dst_idx = v_idx, u_idx
                    
                    # Edge style: flow vs sight-only, bridge vs redundant
                    is_bridge = (u, v) in bridges or (v, u) in bridges
                    
                    if total > 0:
                        sw = max(2, grid.radius * 0.08) + (total / flow.max_flow) * grid.radius * 0.15
                        color = "#4CAF50"
                        dash = ""
                        opacity = edge_opacity
                    elif is_bridge:
                        sw = max(2, grid.radius * 0.08)
                        color = "#E53935"
                        dash = 'stroke-dasharray="5,3"'
                        opacity = edge_opacity * 0.6
                    else:
                        sw = max(1.2, grid.radius * 0.05)
                        color = flag.darkPrimary
                        dash = 'stroke-dasharray="5,3"'
                        opacity = edge_opacity * 0.4
                    
                    for fi_s in ctx.fine_indices(src_idx):
                        for fi_d in ctx.fine_indices(dst_idx):
                            if 0 <= fi_s < N and 0 <= fi_d < N:
                                c1 = grid.hexes[fi_s].center
                                c2 = grid.hexes[fi_d].center
                                dx = c2.x - c1.x
                                dy = c2.y - c1.y
                                dist = math.sqrt(dx*dx + dy*dy)
                                if dist < 1: continue
                                ux, uy = dx / dist, dy / dist
                                shrink = grid.radius * 0.4
                                
                                parts.append(
                                    f'<line x1="{c1.x + ux*shrink:.1f}" y1="{c1.y + uy*shrink:.1f}" '
                                    f'x2="{c2.x - ux*shrink:.1f}" y2="{c2.y - uy*shrink:.1f}" '
                                    f'stroke="{color}" stroke-opacity="{opacity:.2f}" '
                                    f'stroke-width="{sw:.1f}" stroke-linecap="round" '
                                    f'marker-end="url(#{arrow_id})" {dash}/>')
                                
                                # Flow label at midpoint
                                if total > 0:
                                    mx = (c1.x + c2.x) / 2
                                    my = (c1.y + c2.y) / 2
                                    px, py = -uy * grid.radius * 0.4, ux * grid.radius * 0.4
                                    fs = max(8, grid.radius * 0.28)
                                    parts.append(
                                        f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                                        f'text-anchor="middle" font-size="{fs:.0f}" '
                                        f'font-weight="bold" fill="#4CAF50" opacity="0.8">'
                                        f'{total:.0f}🌾</text>')
            
            # --- Target direction indicator ---
            if is_focused:
                target = getattr(squad, '_target', None)
                if target and target.hex_idx >= 0:
                    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
                    if king_id:
                        k_idx = G.nodes[king_id]['hex_idx']
                        t_idx = target.hex_idx
                        for fi_k in ctx.fine_indices(k_idx):
                            if 0 <= fi_k < N:
                                kc = grid.hexes[fi_k].center
                                for fi_t in ctx.fine_indices(t_idx):
                                    if 0 <= fi_t < N:
                                        tc = grid.hexes[fi_t].center
                                        dx = tc.x - kc.x
                                        dy = tc.y - kc.y
                                        dist = math.sqrt(dx*dx + dy*dy)
                                        if dist < 1: break
                                        ux, uy = dx / dist, dy / dist
                                        arrow_len = grid.radius * 2.5
                                        ax = kc.x + ux * arrow_len
                                        ay = kc.y + uy * arrow_len
                                        icon = "⚔️" if target.type == TargetType.KING else "🌾"
                                        
                                        parts.append(
                                            f'<line x1="{kc.x:.1f}" y1="{kc.y:.1f}" '
                                            f'x2="{ax:.1f}" y2="{ay:.1f}" '
                                            f'stroke="{flag.primary}" stroke-opacity="0.4" '
                                            f'stroke-width="{max(3, grid.radius * 0.15):.1f}" '
                                            f'stroke-linecap="round" stroke-dasharray="8,4"/>')
                                        parts.append(
                                            f'<text x="{ax:.1f}" y="{ay:.1f}" '
                                            f'text-anchor="middle" font-size="{max(12, grid.radius * 0.5):.0f}" '
                                            f'dominant-baseline="central">{icon}</text>')
                                        break
            
            # --- Nodes ---
            for nid, data in G.nodes(data=True):
                piece = data['piece']
                idx = data['hex_idx']
                post_net = flow.post_net(nid)
                is_bn = nid in flow.bottlenecks
                
                tenure = piece.place.tenure if piece.place else 0
                life = LIFESPAN.get(piece.type, 999)
                turns_left = max(0, life - tenure) if life < 999 else -1
                
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        c = grid.hexes[fi].center
                        r = grid.radius * 0.25
                        
                        # Bottleneck warning ring
                        if is_bn and is_focused:
                            parts.append(glyphs.ring(
                                c, r * bottleneck_r_mult,
                                glyphs.danger_style()))
                        
                        # Node glyph
                        if post_net > 0:
                            parts.append(glyphs.sheaf(c, size=r * 2.5))
                        elif post_net < 0:
                            parts.append(glyphs.shield(c, size=r * 2))
                        else:
                            parts.append(glyphs.ring(c, r))
                        
                        if is_focused:
                            # Pantry bar
                            pantry = piece.place.pantry if piece.place else 0
                            bar_w = grid.radius * 0.8
                            bar_h = max(3, grid.radius * 0.1)
                            bar_x = c.x - bar_w / 2
                            bar_y = c.y + r * 2.0
                            
                            max_pantry = max(piece.food.cost * 2, 20)
                            fill_frac = max(0, min(1, pantry / max_pantry))
                            fill_color = "#4CAF50" if pantry > 0 else "#E53935"
                            
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="#ddd" opacity="0.6"/>')
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w * fill_frac:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="{fill_color}" opacity="0.8"/>')
                            
                            # Net + turns label
                            label_y = bar_y + bar_h + max(8, grid.radius * 0.3)
                            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
                            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
                            fs = max(8, grid.radius * 0.28)
                            
                            parts.append(
                                f'<text x="{c.x:.1f}" y="{label_y:.1f}" '
                                f'text-anchor="middle" font-size="{fs:.0f}">'
                                f'<tspan fill="{net_color}" font-weight="bold">{post_net:+.0f}</tspan>'
                                f' <tspan fill="#888">{turns_str}</tspan>'
                                f'</text>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)
```

Key changes from the old version:

- **`SquadFlow.from_squad(squad)`** replaces all inline flow computation
- **`flow.edge_flow(u, v)`** determines arrow direction (sources → sinks)
- **`flow.post_net(nid)`** drives node glyphs — sheaf for surplus, shield for deficit, ring for balanced
- **`flow.bottlenecks`** replaces inline `nx.articulation_points` call
- **Edge flow labels** on the map showing food amounts being transferred
- **Flow-weighted edge thickness** — thicker green lines where more food moves
- Node labels now show **post-flow net** instead of raw net